# Charge and Covariance Convergence Diagnostics

This local CPU notebook loads completed perfect-correction runs for $N_x\times N_y=20\times30$, $20\times40$, and $20\times50$. It analyzes charge variance, Born-rule charge traces, boundary/bulk localization, the covariance Frobenius norm in the mixed-state purification ensemble, and an adjacent-cycle covariance product residual in the pure-state streaming ensemble. A separate sparse-snapshot $N_x=16$ pilot uses reduced covariance matrices for spatially resolved wall/bulk convergence. No dynamics are rerun.

In [ ]:
# Local CPU allocation. Set CHARGE_VARIANCE_CPU_RANGE, e.g. `4-7` or `0,2,4,6`.
import os

CPU_RANGE = os.environ.get('CHARGE_VARIANCE_CPU_RANGE', '').strip()

def parse_cpu_range(spec):
    cpus = set()
    for part in spec.split(','):
        part = part.strip()
        if not part:
            continue
        if '-' in part:
            lo, hi = (int(value) for value in part.split('-', 1))
            cpus.update(range(lo, hi + 1))
        else:
            cpus.add(int(part))
    return cpus

if CPU_RANGE:
    requested_cpus = parse_cpu_range(CPU_RANGE)
    allowed_cpus = set(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else requested_cpus
    selected_cpus = requested_cpus & allowed_cpus
    if not selected_cpus:
        raise ValueError(f'CPU range {CPU_RANGE!r} does not overlap allowed CPUs {sorted(allowed_cpus)}')
    if hasattr(os, 'sched_setaffinity'):
        os.sched_setaffinity(0, selected_cpus)
else:
    allowed_cpus = sorted(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else list(range(os.cpu_count() or 1))
    selected_cpus = set(allowed_cpus[:min(4, len(allowed_cpus))])
    if hasattr(os, 'sched_setaffinity'):
        os.sched_setaffinity(0, selected_cpus)

thread_count = max(1, len(selected_cpus))
for variable in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[variable] = str(thread_count)
print(f'Using {thread_count} CPU(s): {sorted(selected_cpus)}')

## Estimator

For each saved Gaussian trajectory, the notebook uses the streamed observable

$$\operatorname{Var}(\hat N_F)=\operatorname{tr}(C-C^2),\qquad C=(G+\mathbf{1})/2.$$

The first panel shows the arithmetic mean over the 100 trajectories. The second panel applies `numpy.gradient` to each trajectory on the saved cycle grid and averages the absolute derivative magnitudes; no rolling average or smoothing is applied.

## Runtime and paths

The repository and Colab bundle roots are discovered from the notebook working directory. Plot and table outputs are written to a dedicated analysis directory.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
from IPython.display import Image, display
from scipy.optimize import curve_fit

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'PROJECT_ADMIN' / 'REPO_POLICY.md').exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'PROJECT_ADMIN' / 'REPO_POLICY.md').exists():
    raise FileNotFoundError('Could not locate repository root containing PROJECT_ADMIN/REPO_POLICY.md')

BUNDLE_ROOT = REPO_ROOT / '00_WORKSPACE' / 'COLAB' / 'colab_charge_fluctuations'
CAMPAIGN_ID = 'N20_multi_geometry_nsh1_dwtrunc1_init-maxmix_S100_cycles-2Ny'
RUNS_ROOT = (
    BUNDLE_ROOT / 'gpu_data' / 'purification_dynamics_maxmix' / 'campaigns' / CAMPAIGN_ID / 'runs'
)
STREAMING_CAMPAIGN_ID = 'N20_selected_nsh1_dwtrunc1_init-default_S100_cycles-2Ny'
STREAMING_RUNS_ROOT = (
    BUNDLE_ROOT / 'gpu_data' / 'streaming_covariance_observables' / 'campaigns' / STREAMING_CAMPAIGN_ID / 'runs'
)
SMALL_SYSTEM_BUNDLE_ROOT = REPO_ROOT / '00_WORKSPACE' / 'COLAB' / 'colab_small_system_testing'
SNAPSHOT_RUNS_ROOT = SMALL_SYSTEM_BUNDLE_ROOT / 'gpu_data' / 'pure_state_covariance_snapshots' / 'runs'
ANALYSIS_ROOT = BUNDLE_ROOT / 'analysis_outputs' / 'total_charge_variance_cycle_derivative' / CAMPAIGN_ID
FIGURE_ROOT = ANALYSIS_ROOT / 'figures'
TABLE_ROOT = ANALYSIS_ROOT / 'tables'
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
TABLE_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Repository root: {REPO_ROOT}')
print(f'Campaign: {CAMPAIGN_ID}')
print(f'Runs root: {RUNS_ROOT}')
print(f'Pure streaming runs root: {STREAMING_RUNS_ROOT}')
print(f'Sparse covariance snapshots root: {SNAPSHOT_RUNS_ROOT}')
print(f'Analysis root: {ANALYSIS_ROOT}')

## Load saved runs and metadata

Only completed `perfect_correction` runs at the three requested geometries are accepted. The checks below require 100 finite trajectories, exact agreement between saved array and summary metadata, and strictly increasing cycle labels.

In [ ]:
TARGET_GEOMETRIES = ((20, 30), (20, 40), (20, 50))
EXPECTED_SAMPLES = 100
ROLLING_WINDOW = 5
ROLLING_MIN_PERIODS = 1
ROLLING_CENTER = True

def centered_rolling_mean_1d(values):
    return (
        pd.Series(np.asarray(values, dtype=np.float64))
        .rolling(window=ROLLING_WINDOW, min_periods=ROLLING_MIN_PERIODS, center=ROLLING_CENTER)
        .mean()
        .to_numpy(dtype=np.float64)
    )

def centered_rolling_mean_rows(values):
    values = np.asarray(values, dtype=np.float64)
    return np.stack([centered_rolling_mean_1d(row) for row in values], axis=0)

records = []

for nx, ny in TARGET_GEOMETRIES:
    run_name = f'N{nx}x{ny}_nsh1_init-maxmix_perfect_correction'
    run_dir = RUNS_ROOT / run_name
    variance_path = run_dir / 'total_charge_variance.npz'
    charge_trace_path = run_dir / 'local_charge_cell_mean.npz'
    local_variance_path = run_dir / 'local_charge_cell_variance.npz'
    summary_path = run_dir / 'run_summary.json'
    if not variance_path.exists() or not charge_trace_path.exists() or not local_variance_path.exists() or not summary_path.exists():
        raise FileNotFoundError(f'Incomplete saved run: {run_dir}')

    summary = json.loads(summary_path.read_text())
    with np.load(variance_path, allow_pickle=False) as saved:
        cycle_labels = saved['cycle_labels'].astype(np.float64, copy=True)
        sample_indices = saved['sample_indices'].astype(np.int64, copy=True)
        charge_variance = saved['total_charge_variance'].astype(np.float64, copy=True)
        formula = str(saved['formula_total_charge_variance'].item())
        config = json.loads(str(saved['config_json'].item()))
    with np.load(charge_trace_path, allow_pickle=False) as saved:
        trace_cycle_labels = saved['cycle_labels'].astype(np.float64, copy=True)
        trace_sample_indices = saved['sample_indices'].astype(np.int64, copy=True)
        local_charge_cell_mean = saved['local_charge_cell_mean'].astype(np.float64, copy=True)
        trace_convention = str(saved['convention'].item())
    with np.load(local_variance_path, allow_pickle=False) as saved:
        local_charge_cell_variance = saved['local_charge_cell_variance'].astype(np.float64, copy=True)
        local_variance_convention = str(saved['convention'].item())

    expected_shape = (EXPECTED_SAMPLES, 2 * ny)
    if charge_variance.shape != expected_shape:
        raise ValueError(f'{run_name}: expected shape {expected_shape}, got {charge_variance.shape}')
    expected_trace_shape = (EXPECTED_SAMPLES, 2 * ny, nx, ny)
    if local_charge_cell_mean.shape != expected_trace_shape:
        raise ValueError(f'{run_name}: expected charge-field shape {expected_trace_shape}, got {local_charge_cell_mean.shape}')
    if local_charge_cell_variance.shape != expected_trace_shape:
        raise ValueError(f'{run_name}: expected local-variance shape {expected_trace_shape}, got {local_charge_cell_variance.shape}')
    if not np.array_equal(trace_cycle_labels, cycle_labels) or not np.array_equal(trace_sample_indices, sample_indices):
        raise ValueError(f'{run_name}: charge-field and variance coordinates disagree')
    if sample_indices.size != EXPECTED_SAMPLES or not np.all(np.isfinite(charge_variance)):
        raise ValueError(f'{run_name}: missing samples or non-finite charge variance values')
    if not np.all(np.diff(cycle_labels) > 0):
        raise ValueError(f'{run_name}: cycle labels are not strictly increasing')
    if (int(summary['Nx']), int(summary['Ny'])) != (nx, ny):
        raise ValueError(f'{run_name}: run-summary geometry mismatch')
    if summary['protocol'] != 'perfect_correction' or config['protocol'] != 'perfect_correction':
        raise ValueError(f'{run_name}: expected perfect_correction protocol')

    derivative = np.gradient(charge_variance, cycle_labels, axis=1, edge_order=2)
    absolute_derivative = np.abs(derivative)
    total_modes = 2.0 * float(nx * ny)
    frob_radicand = total_modes - 4.0 * charge_variance
    if np.min(frob_radicand) < -1e-10:
        raise ValueError(f'{run_name}: charge variance implies a negative Frobenius-norm radicand')
    frob_norm = np.sqrt(np.maximum(frob_radicand, 0.0))
    frob_ceiling = np.sqrt(total_modes)
    frob_derivative = np.gradient(frob_norm, cycle_labels, axis=1, edge_order=2)
    frob_absolute_derivative = np.abs(frob_derivative)
    frob_absolute_derivative_mean = frob_absolute_derivative.mean(axis=0)
    total_charge_trace = local_charge_cell_mean.sum(axis=(2, 3))
    normalized_trace_offset = (total_charge_trace - float(nx * ny)) / total_modes
    rolling_normalized_trace_offset = centered_rolling_mean_rows(normalized_trace_offset)
    normalized_trace_variance = total_charge_trace.var(axis=0, ddof=0) / total_modes
    late_mask = cycle_labels > float(ny)
    wall_positions = np.asarray(config['dw_loc'], dtype=np.int64)
    x_coords = np.arange(nx, dtype=np.int64)
    distance_to_wall = np.min(np.abs(x_coords[:, None] - wall_positions[None, :]), axis=1)
    ensemble_mean_field = local_charge_cell_mean.mean(axis=0)
    occupancy_bias_x = np.abs(ensemble_mean_field[late_mask] - 1.0).mean(axis=(0, 2))
    trajectory_variance_x = local_charge_cell_mean[:, late_mask].var(axis=0, ddof=0).mean(axis=(0, 2))
    quantum_local_variance_x = local_charge_cell_variance[:, late_mask].mean(axis=(0, 1, 3))
    records.append({
        'Nx': nx,
        'Ny': ny,
        'label': rf'$N_x\times N_y={nx}\times {ny}$',
        'cycles': cycle_labels,
        'sample_indices': sample_indices,
        'charge_variance': charge_variance,
        'charge_variance_mean': charge_variance.mean(axis=0),
        'charge_variance_sem': charge_variance.std(axis=0, ddof=1) / np.sqrt(EXPECTED_SAMPLES),
        'derivative': derivative,
        'absolute_derivative': absolute_derivative,
        'absolute_derivative_mean': absolute_derivative.mean(axis=0),
        'absolute_derivative_sem': absolute_derivative.std(axis=0, ddof=1) / np.sqrt(EXPECTED_SAMPLES),
        'frob_norm': frob_norm,
        'frob_norm_mean': frob_norm.mean(axis=0),
        'frob_norm_sem': frob_norm.std(axis=0, ddof=1) / np.sqrt(EXPECTED_SAMPLES),
        'frob_norm_per_mode_mean': frob_norm.mean(axis=0) / total_modes,
        'frob_norm_per_mode_sem': frob_norm.std(axis=0, ddof=1) / (np.sqrt(EXPECTED_SAMPLES) * total_modes),
        'frob_norm_rms_mean': frob_norm.mean(axis=0) / np.sqrt(total_modes),
        'frob_norm_rms_sem': frob_norm.std(axis=0, ddof=1) / np.sqrt(EXPECTED_SAMPLES * total_modes),
        'frob_ceiling': frob_ceiling,
        'frob_ceiling_per_mode': frob_ceiling / total_modes,
        'frob_ceiling_rms': 1.0,
        'frob_normalized_deficit_mean': (1.0 - frob_norm / frob_ceiling).mean(axis=0),
        'frob_derivative': frob_derivative,
        'frob_absolute_derivative': frob_absolute_derivative,
        'frob_absolute_derivative_mean': frob_absolute_derivative_mean,
        'frob_absolute_derivative_sem': frob_absolute_derivative.std(axis=0, ddof=1) / np.sqrt(EXPECTED_SAMPLES),
        'rolling_frob_absolute_derivative_mean': centered_rolling_mean_1d(frob_absolute_derivative_mean),
        'frob_absolute_derivative_per_mode_mean': frob_absolute_derivative_mean / total_modes,
        'frob_absolute_derivative_per_mode_sem': frob_absolute_derivative.std(axis=0, ddof=1) / (np.sqrt(EXPECTED_SAMPLES) * total_modes),
        'rolling_frob_absolute_derivative_per_mode_mean': centered_rolling_mean_1d(frob_absolute_derivative_mean) / total_modes,
        'frob_absolute_derivative_rms_mean': frob_absolute_derivative_mean / np.sqrt(total_modes),
        'frob_absolute_derivative_rms_sem': frob_absolute_derivative.std(axis=0, ddof=1) / np.sqrt(EXPECTED_SAMPLES * total_modes),
        'rolling_frob_absolute_derivative_rms_mean': centered_rolling_mean_1d(frob_absolute_derivative_mean) / np.sqrt(total_modes),
        'total_charge_trace': total_charge_trace,
        'total_charge_trace_mean': total_charge_trace.mean(axis=0),
        'total_charge_trace_sem': total_charge_trace.std(axis=0, ddof=1) / np.sqrt(EXPECTED_SAMPLES),
        'total_charge_trace_population_variance': total_charge_trace.var(axis=0, ddof=0),
        'normalized_trace_offset_mean': normalized_trace_offset.mean(axis=0),
        'normalized_trace_offset_sem': normalized_trace_offset.std(axis=0, ddof=1) / np.sqrt(EXPECTED_SAMPLES),
        'rolling_normalized_trace_offset_mean': rolling_normalized_trace_offset.mean(axis=0),
        'rolling_normalized_trace_offset_sem': rolling_normalized_trace_offset.std(axis=0, ddof=1) / np.sqrt(EXPECTED_SAMPLES),
        'normalized_trace_population_variance': normalized_trace_variance,
        'rolling_normalized_trace_population_variance': centered_rolling_mean_1d(normalized_trace_variance),
        'half_filling_trace': float(nx * ny),
        'trace_convention': trace_convention,
        'local_variance_convention': local_variance_convention,
        'late_cycle_min': int(cycle_labels[late_mask][0]),
        'late_cycle_max': int(cycle_labels[late_mask][-1]),
        'distance_to_wall': distance_to_wall,
        'occupancy_bias_x': occupancy_bias_x,
        'trajectory_variance_x': trajectory_variance_x,
        'quantum_local_variance_x': quantum_local_variance_x,
        'formula': formula,
        'config': config,
        'summary': summary,
        'variance_path': variance_path,
        'charge_trace_path': charge_trace_path,
        'local_variance_path': local_variance_path,
    })

parameter_df = pd.DataFrame([
    {
        'geometry': f"N{record['Nx']}x{record['Ny']}",
        'samples': record['charge_variance'].shape[0],
        'cycles': record['charge_variance'].shape[1],
        'cycle_min': record['cycles'][0],
        'cycle_max': record['cycles'][-1],
        'protocol': record['config']['protocol'],
        'init_mode': record['config']['init_mode'],
        'nshell': record['config']['nshell'],
        'dw_truncation': record['config']['dw_truncation'],
        'canonical_entry_point': record['config']['canonical_dynamics_entry_point'],
        'variance_source': str(record['variance_path'].relative_to(REPO_ROOT)),
        'charge_trace_source': str(record['charge_trace_path'].relative_to(REPO_ROOT)),
        'local_variance_source': str(record['local_variance_path'].relative_to(REPO_ROOT)),
    }
    for record in records
])
display(parameter_df)

## Charge variance and cycle derivative

The left panel tests charge sharpening directly. The right panel resolves the absolute rate at which the variance changes per cycle. Both panels use logarithmic $y$-axes. Bands show the standard error over trajectories and are often narrower than the plotted line.

In [ ]:
font_name = 'CMU Sans Serif' if any(font.name == 'CMU Sans Serif' for font in font_manager.fontManager.ttflist) else 'DejaVu Sans'
style = {
    'font.family': font_name,
    'font.size': 8,
    'axes.labelsize': 8,
    'axes.titlesize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'lines.linewidth': 1.25,
    'axes.linewidth': 0.6,
}
colors = ('#0072B2', '#D55E00', '#009E73')
png_path = FIGURE_ROOT / 'total_charge_variance_and_cycle_derivative.png'
pdf_path = FIGURE_ROOT / 'total_charge_variance_and_cycle_derivative.pdf'

with plt.rc_context(style):
    fig, axes = plt.subplots(1, 2, figsize=(6.75, 2.65), constrained_layout=True)
    for color, record in zip(colors, records):
        cycles = record['cycles']
        mean = record['charge_variance_mean']
        sem = record['charge_variance_sem']
        derivative_mean = record['absolute_derivative_mean']
        derivative_sem = record['absolute_derivative_sem']

        axes[0].plot(cycles, mean, color=color, label=record['label'])
        axes[0].fill_between(cycles, mean - sem, mean + sem, color=color, alpha=0.18, linewidth=0)
        axes[1].plot(cycles, derivative_mean, color=color, label=record['label'])
        axes[1].fill_between(
            cycles, derivative_mean - derivative_sem, derivative_mean + derivative_sem,
            color=color, alpha=0.18, linewidth=0,
        )

    axes[0].set_xlabel('cycle')
    axes[0].set_ylabel(r'$\langle\operatorname{Var}(\hat N_F)\rangle_{\rm traj}$')
    axes[0].set_yscale('log')
    axes[0].set_title('(a) Total charge variance')
    axes[1].set_xlabel('cycle')
    axes[1].set_ylabel(r'$\langle|\mathrm{d}\operatorname{Var}(\hat N_F)/\mathrm{d}(\mathrm{cycle})|\rangle_{\rm traj}$')
    axes[1].set_yscale('log')
    axes[1].set_title('(b) Absolute cycle derivative')

    for ax in axes:
        ax.grid(True, alpha=0.22, linewidth=0.4)
        ax.legend(frameon=False, loc='best')

    fig.savefig(png_path, dpi=300, bbox_inches='tight')
    fig.savefig(pdf_path, dpi=300, bbox_inches='tight')
    plt.show()

print(f'Saved PNG: {png_path}')
print(f'Saved PDF: {pdf_path}')

## Covariance Frobenius norm and convergence rate

The saved purification runs did not store covariance histories, but their raw Frobenius norms are recovered exactly from the saved total charge variance. With $C=(G+I)/2$, $V=2N_xN_y$, and Hermitian $G$,

$$\operatorname{Var}(\hat N_F)=\operatorname{Tr}(C-C^2)=\frac{V-\operatorname{Tr}(G^2)}{4}=\frac{V-\lVert G\rVert_F^2}{4}.$$

Thus $\lVert G\rVert_F=\sqrt{V-4\operatorname{Var}(\hat N_F)}$ trajectory by trajectory. Since a pure covariance satisfies $G^2=I$ and therefore $\lVert G\rVert_F=\sqrt V$, the size-collapsing normalization is $\lVert G\rVert_F/\sqrt V$, whose pure-state ceiling is 1. The right panel applies the same $\sqrt V$ normalization to $|\mathrm d\lVert G\rVert_F/\mathrm d(\mathrm{cycle})|$ on a logarithmic axis; faint curves and bands are unsmoothed, while thick curves are centered rolling means with $w=5$. This scalar derivative measures the rate of purification. It is weaker than $\lVert G_t-G_{t-1}\rVert_F$, which would also detect motion at constant norm and cannot be reconstructed from these scalar files.

In [ ]:
frob_png_path = FIGURE_ROOT / 'frobenius_norm_and_cycle_derivative.png'
frob_pdf_path = FIGURE_ROOT / 'frobenius_norm_and_cycle_derivative.pdf'

with plt.rc_context(style):
    fig, axes = plt.subplots(1, 2, figsize=(6.75, 2.65), constrained_layout=True)
    for color, record in zip(colors, records):
        cycles = record['cycles']
        norm_mean = record['frob_norm_rms_mean']
        norm_sem = record['frob_norm_rms_sem']
        rate_mean = record['frob_absolute_derivative_rms_mean']
        rate_sem = record['frob_absolute_derivative_rms_sem']
        rolling_rate = record['rolling_frob_absolute_derivative_rms_mean']

        axes[0].plot(cycles, norm_mean, color=color, label=record['label'])
        axes[0].fill_between(cycles, norm_mean - norm_sem, norm_mean + norm_sem, color=color, alpha=0.18, linewidth=0)
        axes[0].axhline(record['frob_ceiling_rms'], color=color, linestyle=':', linewidth=0.8, alpha=0.65)
        axes[1].plot(cycles, rate_mean, color=color, alpha=0.22, linewidth=0.7)
        axes[1].fill_between(
            cycles, np.maximum(rate_mean - rate_sem, np.finfo(float).tiny), rate_mean + rate_sem,
            color=color, alpha=0.12, linewidth=0,
        )
        axes[1].plot(cycles, rolling_rate, color=color, linewidth=1.45, label=record['label'])

    axes[0].set_xlabel('cycle')
    axes[0].set_ylabel(r'$\langle\Vert G\Vert_F\rangle_{\rm traj}/\sqrt{2N_xN_y}$')
    axes[0].set_title('(a) RMS-normalized covariance norm')
    axes[1].set_xlabel('cycle')
    axes[1].set_ylabel(r'$\langle|\mathrm{d}\Vert G\Vert_F/\mathrm{d}(\mathrm{cycle})|\rangle_{\rm traj}/\sqrt{2N_xN_y}$')
    axes[1].set_yscale('log')
    axes[1].set_title(f'(b) Absolute rate, rolling $w={ROLLING_WINDOW}$')

    for ax in axes:
        ax.grid(True, alpha=0.22, linewidth=0.4)
        ax.legend(frameon=False, loc='best')

    fig.savefig(frob_png_path, dpi=300, bbox_inches='tight')
    fig.savefig(frob_pdf_path, dpi=300, bbox_inches='tight')
    plt.show()

print(f'Saved PNG: {frob_png_path}')
print(f'Saved PDF: {frob_pdf_path}')

## Adjacent-cycle covariance product residual

The pure-state streaming campaign (`init_mode=default`) stores `frob_successive_delta` for all 100 trajectories even though it does not retain full covariance histories. For its Hermitian involutory covariances, $G_t^2=I$,

$$G_tG_{t+1}-I=G_t(G_{t+1}-G_t),\qquad \lVert G_tG_{t+1}-I\rVert_F=\lVert G_{t+1}-G_t\rVert_F.$$

The equality follows because left multiplication by $G_t$ preserves the Frobenius norm. Therefore the saved successive-distance scalar is exactly the requested product residual. Cycle $t$ below labels the transition ending at saved cycle $t$ (cycle 1 compares the initial state with the first completed cycle). Both panels use logarithmic $y$-axes. Thick curves use a centered rolling mean with $w=5$; faint curves and bands show the unsmoothed trajectory mean and its standard error. This pure-state ensemble is separate from the `init-maxmix` purification ensemble used above. The metric is global: testing whether its persistent part is specifically wall-localized would require a restricted wall/bulk covariance observer or saved matrices, neither of which exists for these production runs.

In [ ]:
product_records = []
for nx, ny in TARGET_GEOMETRIES:
    run_name = f'N{nx}x{ny}_nsh1_init-default_perfect_correction'
    run_dir = STREAMING_RUNS_ROOT / run_name
    scalar_path = run_dir / 'scalar_metrics.csv'
    summary_path = run_dir / 'run_summary.json'
    if not scalar_path.exists() or not summary_path.exists():
        raise FileNotFoundError(f'Incomplete pure streaming run: {run_dir}')

    summary = json.loads(summary_path.read_text())
    metrics = pd.read_csv(scalar_path)
    required_columns = {'sample_index', 'cycle_label', 'frob_norm', 'frob_successive_delta'}
    if not required_columns.issubset(metrics.columns):
        raise ValueError(f'{run_name}: missing scalar columns {sorted(required_columns - set(metrics.columns))}')
    if summary['init_mode'] != 'default' or summary['protocol'] != 'perfect_correction':
        raise ValueError(f'{run_name}: expected pure default perfect-correction run')
    if summary['covariance_history_saved']:
        raise ValueError(f'{run_name}: analysis assumes the documented scalar-only streaming layout')

    residual_frame = metrics.pivot(index='sample_index', columns='cycle_label', values='frob_successive_delta').sort_index().sort_index(axis=1)
    norm_frame = metrics.pivot(index='sample_index', columns='cycle_label', values='frob_norm').sort_index().sort_index(axis=1)
    product_cycles = residual_frame.columns.to_numpy(dtype=np.float64)
    product_residual = residual_frame.to_numpy(dtype=np.float64)
    streamed_norm = norm_frame.to_numpy(dtype=np.float64)
    expected_shape = (EXPECTED_SAMPLES, 2 * ny)
    if product_residual.shape != expected_shape or not np.all(np.isfinite(product_residual)):
        raise ValueError(f'{run_name}: expected finite product-residual shape {expected_shape}, got {product_residual.shape}')
    pure_norm = np.sqrt(2.0 * nx * ny)
    if not np.allclose(streamed_norm, pure_norm, rtol=1e-10, atol=1e-10):
        raise ValueError(f'{run_name}: covariance norm does not satisfy the pure-state involution check')

    product_derivative = np.gradient(product_residual, product_cycles, axis=1, edge_order=2)
    product_absolute_derivative = np.abs(product_derivative)
    product_mean = product_residual.mean(axis=0)
    product_rate_mean = product_absolute_derivative.mean(axis=0)
    product_records.append({
        'Nx': nx,
        'Ny': ny,
        'label': rf'$N_x\times N_y={nx}\times {ny}$',
        'cycles': product_cycles,
        'product_residual': product_residual,
        'product_residual_mean': product_mean,
        'product_residual_sem': product_residual.std(axis=0, ddof=1) / np.sqrt(EXPECTED_SAMPLES),
        'rolling_product_residual_mean': centered_rolling_mean_1d(product_mean),
        'product_absolute_derivative': product_absolute_derivative,
        'product_absolute_derivative_mean': product_rate_mean,
        'product_absolute_derivative_sem': product_absolute_derivative.std(axis=0, ddof=1) / np.sqrt(EXPECTED_SAMPLES),
        'rolling_product_absolute_derivative_mean': centered_rolling_mean_1d(product_rate_mean),
        'pure_frobenius_scale': pure_norm,
        'scalar_path': scalar_path,
        'summary_path': summary_path,
    })

product_png_path = FIGURE_ROOT / 'adjacent_cycle_covariance_product_residual_and_derivative.png'
product_pdf_path = FIGURE_ROOT / 'adjacent_cycle_covariance_product_residual_and_derivative.pdf'
with plt.rc_context(style):
    fig, axes = plt.subplots(1, 2, figsize=(6.75, 2.65), constrained_layout=True)
    for color, record in zip(colors, product_records):
        cycles = record['cycles']
        residual_mean = record['product_residual_mean']
        residual_sem = record['product_residual_sem']
        rate_mean = record['product_absolute_derivative_mean']
        rate_sem = record['product_absolute_derivative_sem']

        axes[0].plot(cycles, residual_mean, color=color, alpha=0.22, linewidth=0.7)
        axes[0].fill_between(cycles, residual_mean - residual_sem, residual_mean + residual_sem, color=color, alpha=0.12, linewidth=0)
        axes[0].plot(cycles, record['rolling_product_residual_mean'], color=color, linewidth=1.45, label=record['label'])
        axes[1].plot(cycles, rate_mean, color=color, alpha=0.22, linewidth=0.7)
        axes[1].fill_between(
            cycles, np.maximum(rate_mean - rate_sem, np.finfo(float).tiny), rate_mean + rate_sem,
            color=color, alpha=0.12, linewidth=0,
        )
        axes[1].plot(cycles, record['rolling_product_absolute_derivative_mean'], color=color, linewidth=1.45, label=record['label'])

    axes[0].set_xlabel('cycle $t$ (transition ending at $t$)')
    axes[0].set_ylabel(r'$\langle\Vert G_{t-1}G_t-I\Vert_F\rangle_{\rm traj}$')
    axes[0].set_yscale('log')
    axes[0].set_title(f'(a) Product residual, rolling $w={ROLLING_WINDOW}$')
    axes[1].set_xlabel('cycle $t$')
    axes[1].set_ylabel(r'$\langle|\mathrm{d}\Vert G_{t-1}G_t-I\Vert_F/\mathrm{d}t|\rangle_{\rm traj}$')
    axes[1].set_yscale('log')
    axes[1].set_title(f'(b) Absolute derivative, rolling $w={ROLLING_WINDOW}$')
    for ax in axes:
        ax.grid(True, alpha=0.22, linewidth=0.4)
        ax.legend(frameon=False, loc='best')
    fig.savefig(product_png_path, dpi=300, bbox_inches='tight')
    fig.savefig(product_pdf_path, dpi=300, bbox_inches='tight')
    plt.show()

print(f'Saved PNG: {product_png_path}')
print(f'Saved PDF: {product_pdf_path}')

## Spatially resolved reduced-covariance convergence pilot

Full covariance histories were not saved for the $N_x=20$ production runs, but a separate pure-state snapshot campaign contains 10 trajectories for $N_x=16$, $N_y=30,40$, `nshell=1`, at cycles $5,10,20,50$. These runs use the same perfect-correction protocol with `dw_truncation=True`; the canonical domain-wall constructor places the walls at $x=4,12$. This section is therefore a sparse-cycle pilot, not a like-for-like replacement for the production ensemble.

For each $x$, $G_x(t)$ is the principal covariance submatrix containing both orbitals at that $x$ for every $y$—$2N_y$ modes total. For a fermionic Gaussian state this is precisely the covariance of the reduced state after tracing out all other $x$ rows. The temporal statistic is

$$R_x(t_a,t_b)=\frac{\lVert G_x(t_b)-G_x(t_a)\rVert_F}{(t_b-t_a)\sqrt{2N_y}}.$$

Both quantities use the RMS spectral normalization $\sqrt{2N_y}$. Because the reduced state is generally mixed, the second row separately reports $M_x(t)=\lVert G_x(t)^2-I_x\rVert_F/\sqrt{2N_y}$. This is why the product residual $\lVert G_x(t_a)G_x(t_b)-I_x\rVert_F$ is not used locally: it would remain nonzero even for an unchanged mixed reduced state.

In [ ]:
SPATIAL_PILOT_NX = 16
SPATIAL_PILOT_NYS = (30, 40)
SPATIAL_PILOT_NSHELL = 1
SPATIAL_PILOT_SAMPLES = 10
SPATIAL_PILOT_CYCLES = (5, 10, 20, 50)
SPATIAL_DISPLAY_FLOOR = 1e-12

def x_row_mode_indices(nx, ny, x):
    return np.asarray([2 * x + 2 * nx * y + orbital for y in range(ny) for orbital in range(2)], dtype=np.int64)

def sem_series(values):
    return values.std(ddof=1) / np.sqrt(values.count())

spatial_covariance_rows = []
spatial_snapshot_sources = []
for ny in SPATIAL_PILOT_NYS:
    run_name = f'N{SPATIAL_PILOT_NX}x{ny}_nsh{SPATIAL_PILOT_NSHELL}_perfect_correction'
    latest_path = SNAPSHOT_RUNS_ROOT / run_name / 'latest_run.json'
    if not latest_path.exists():
        raise FileNotFoundError(latest_path)
    metadata = json.loads(latest_path.read_text())
    run_dir = SMALL_SYSTEM_BUNDLE_ROOT / 'gpu_data' / metadata['run_dir_relative']
    snapshot_paths = sorted(run_dir.glob('batch_*_snapshots.npy'))
    if len(snapshot_paths) != 1:
        raise ValueError(f'{run_name}: expected one complete snapshot shard, found {len(snapshot_paths)}')
    snapshot_path = snapshot_paths[0]
    snapshots = np.load(snapshot_path, mmap_mode='r')
    cycles = np.asarray(metadata['snapshot_cycles'], dtype=np.int64)
    expected_shape = (SPATIAL_PILOT_SAMPLES, len(SPATIAL_PILOT_CYCLES), 2 * SPATIAL_PILOT_NX * ny, 2 * SPATIAL_PILOT_NX * ny)
    if snapshots.shape != expected_shape or tuple(cycles) != SPATIAL_PILOT_CYCLES:
        raise ValueError(f'{run_name}: unexpected snapshot shape or cycle labels')
    if metadata['protocol'] != 'perfect_correction' or metadata['init_mode'] != 'default' or not metadata['dw_truncation']:
        raise ValueError(f'{run_name}: unexpected snapshot protocol metadata')

    spatial_snapshot_sources.extend([latest_path, snapshot_path])
    for sample_index in range(SPATIAL_PILOT_SAMPLES):
        for x in range(SPATIAL_PILOT_NX):
            mode_indices = x_row_mode_indices(SPATIAL_PILOT_NX, ny, x)
            mode_count = mode_indices.size
            restricted_covariances = []
            for snapshot_index, cycle in enumerate(cycles):
                sub_g = np.asarray(snapshots[sample_index, snapshot_index][np.ix_(mode_indices, mode_indices)])
                restricted_covariances.append(sub_g)
                identity = np.eye(mode_count, dtype=sub_g.dtype)
                mixedness = np.linalg.norm(sub_g @ sub_g - identity, ord='fro') / np.sqrt(mode_count)
                spatial_covariance_rows.append({
                    'Nx': SPATIAL_PILOT_NX, 'Ny': ny, 'sample_index': sample_index, 'x': x,
                    'metric': 'reduced_covariance_mixedness',
                    'cycle_start': int(cycle), 'cycle_end': int(cycle),
                    'value_rms_normalized': mixedness,
                })
            for later_index in range(1, len(cycles)):
                cycle_start = int(cycles[later_index - 1])
                cycle_end = int(cycles[later_index])
                delta_cycle = cycle_end - cycle_start
                rate = np.linalg.norm(
                    restricted_covariances[later_index] - restricted_covariances[later_index - 1], ord='fro'
                ) / (delta_cycle * np.sqrt(mode_count))
                spatial_covariance_rows.append({
                    'Nx': SPATIAL_PILOT_NX, 'Ny': ny, 'sample_index': sample_index, 'x': x,
                    'metric': 'reduced_covariance_temporal_rate',
                    'cycle_start': cycle_start, 'cycle_end': cycle_end,
                    'value_rms_normalized': rate,
                })

spatial_covariance_raw_df = pd.DataFrame(spatial_covariance_rows)
spatial_covariance_profile_df = (
    spatial_covariance_raw_df
    .groupby(['Nx', 'Ny', 'metric', 'cycle_start', 'cycle_end', 'x'], as_index=False)
    .agg(mean_rms_normalized=('value_rms_normalized', 'mean'), sem_rms_normalized=('value_rms_normalized', sem_series), trajectories=('sample_index', 'nunique'))
)

wall_positions_pilot = np.asarray([4, 12], dtype=np.int64)
distance_to_wall_pilot = np.min(np.abs(np.arange(SPATIAL_PILOT_NX)[:, None] - wall_positions_pilot[None, :]), axis=1)
def spatial_region(x):
    if 4 <= x <= 12 and distance_to_wall_pilot[x] <= 1:
        return 'active_wall_rows'
    if 4 <= x <= 12:
        return 'active_bulk_rows'
    return 'truncated_exterior'

spatial_covariance_raw_df['region'] = spatial_covariance_raw_df['x'].map(spatial_region)
spatial_sample_region_df = (
    spatial_covariance_raw_df
    .groupby(['Nx', 'Ny', 'metric', 'cycle_start', 'cycle_end', 'sample_index', 'region'], as_index=False)
    .agg(sample_region_mean_rms_normalized=('value_rms_normalized', 'mean'))
)
spatial_covariance_region_df = (
    spatial_sample_region_df
    .groupby(['Nx', 'Ny', 'metric', 'cycle_start', 'cycle_end', 'region'], as_index=False)
    .agg(mean_rms_normalized=('sample_region_mean_rms_normalized', 'mean'), sem_rms_normalized=('sample_region_mean_rms_normalized', sem_series), trajectories=('sample_index', 'nunique'))
)
rate_region_wide = (
    spatial_covariance_region_df[spatial_covariance_region_df['metric'] == 'reduced_covariance_temporal_rate']
    .pivot(index=['Nx', 'Ny', 'cycle_start', 'cycle_end'], columns='region', values='mean_rms_normalized')
    .reset_index()
)
rate_region_wide['active_wall_to_bulk_rate_ratio'] = rate_region_wide['active_wall_rows'] / rate_region_wide['active_bulk_rows']

spatial_covariance_png_path = FIGURE_ROOT / 'spatial_reduced_covariance_convergence_pilot.png'
spatial_covariance_pdf_path = FIGURE_ROOT / 'spatial_reduced_covariance_convergence_pilot.pdf'
interval_colors = ('#0072B2', '#D55E00', '#009E73')
cycle_colors = ('#56B4E9', '#0072B2', '#D55E00', '#009E73')
with plt.rc_context(style):
    fig, axes = plt.subplots(2, 2, figsize=(6.75, 5.0), constrained_layout=True, sharex=True)
    for column, ny in enumerate(SPATIAL_PILOT_NYS):
        rate_profile = spatial_covariance_profile_df[(spatial_covariance_profile_df['Ny'] == ny) & (spatial_covariance_profile_df['metric'] == 'reduced_covariance_temporal_rate')]
        mixed_profile = spatial_covariance_profile_df[(spatial_covariance_profile_df['Ny'] == ny) & (spatial_covariance_profile_df['metric'] == 'reduced_covariance_mixedness')]
        for color, ((cycle_start, cycle_end), group) in zip(interval_colors, rate_profile.groupby(['cycle_start', 'cycle_end'], sort=True)):
            x = group['x'].to_numpy()
            mean = np.maximum(group['mean_rms_normalized'].to_numpy(), SPATIAL_DISPLAY_FLOOR)
            sem = group['sem_rms_normalized'].to_numpy()
            axes[0, column].plot(x, mean, color=color, marker='o', markersize=2.2, label=f'{cycle_start}–{cycle_end}')
            axes[0, column].fill_between(x, np.maximum(mean - sem, SPATIAL_DISPLAY_FLOOR), mean + sem, color=color, alpha=0.14, linewidth=0)
        for color, (cycle, group) in zip(cycle_colors, mixed_profile.groupby('cycle_end', sort=True)):
            x = group['x'].to_numpy()
            mean = np.maximum(group['mean_rms_normalized'].to_numpy(), SPATIAL_DISPLAY_FLOOR)
            sem = group['sem_rms_normalized'].to_numpy()
            axes[1, column].plot(x, mean, color=color, marker='o', markersize=2.2, label=f'cycle {cycle}')
            axes[1, column].fill_between(x, np.maximum(mean - sem, SPATIAL_DISPLAY_FLOOR), mean + sem, color=color, alpha=0.12, linewidth=0)
        for row in range(2):
            axes[row, column].axvspan(4, 12, color='0.5', alpha=0.08, linewidth=0, zorder=0)
            axes[row, column].axvline(4, color='0.35', linestyle=':', linewidth=0.75, zorder=1)
            axes[row, column].axvline(12, color='0.35', linestyle=':', linewidth=0.75, zorder=1)
            axes[row, column].set_yscale('log')
            axes[row, column].set_xticks((0, 4, 8, 12, 15))
            axes[row, column].grid(True, alpha=0.22, linewidth=0.4)
        axes[0, column].set_title(rf'$N_x\times N_y=16\times {ny}$')
        axes[1, column].set_xlabel('$x$ row retained after partial trace')
    axes[0, 0].set_ylabel(r'$\langle R_x(t_a,t_b)\rangle_{\rm traj}$')
    axes[1, 0].set_ylabel(r'$\langle\Vert G_x^2-I_x\Vert_F/\sqrt{2N_y}\rangle_{\rm traj}$')
    axes[0, 0].legend(frameon=False, title='cycle interval', loc='best')
    axes[1, 0].legend(frameon=False, loc='best')
    fig.savefig(spatial_covariance_png_path, dpi=300, bbox_inches='tight')
    fig.savefig(spatial_covariance_pdf_path, dpi=300, bbox_inches='tight')
    plt.show()

display(rate_region_wide)
print(f'Saved PNG: {spatial_covariance_png_path}')
print(f'Saved PDF: {spatial_covariance_pdf_path}')

## Quantitative transient-time estimate

We define an operational global convergence time $T_{\mathrm{conv}}$ as the later of two sustained threshold crossings: the ensemble-mean normalized Frobenius deficit remains below $10^{-3}$ and the centered five-cycle RMS-normalized Frobenius-rate remains below $10^{-4}$ per cycle. Trajectory bootstrap resampling gives 95% intervals for these crossings. Separately, the wall-resolved temporal rate is fit in log space to a power law $A t^{-\alpha}$ and an exponential $A e^{-t/\tau}$ using the interval midpoints. The fit comparison is descriptive because the spatial pilot has only three time intervals and ten trajectories per geometry.


In [ ]:
BOOTSTRAP_REPS = 2000
BOOTSTRAP_SEED = 271828
GLOBAL_DEFICIT_THRESHOLD = 1e-3
GLOBAL_RATE_THRESHOLD = 1e-4
rng = np.random.default_rng(BOOTSTRAP_SEED)

def sustained_crossing_cycle(values, cycles, threshold):
    below = np.asarray(values) <= threshold
    sustained = np.logical_and.accumulate(below[::-1])[::-1]
    indices = np.flatnonzero(sustained)
    return float(cycles[indices[0]]) if indices.size else np.nan

def percentile_interval(values):
    finite = np.asarray(values, dtype=np.float64)
    finite = finite[np.isfinite(finite)]
    return tuple(np.percentile(finite, [2.5, 97.5])) if finite.size else (np.nan, np.nan)

global_transient_rows = []
for record in records:
    cycles = record['cycles']
    mode_scale = np.sqrt(2.0 * record['Nx'] * record['Ny'])
    deficits = 1.0 - record['frob_norm'] / record['frob_ceiling']
    rates = record['frob_absolute_derivative'] / mode_scale
    deficit_point = sustained_crossing_cycle(deficits.mean(axis=0), cycles, GLOBAL_DEFICIT_THRESHOLD)
    rate_point = sustained_crossing_cycle(centered_rolling_mean_1d(rates.mean(axis=0)), cycles, GLOBAL_RATE_THRESHOLD)
    joint_point = max(deficit_point, rate_point)
    deficit_boot, rate_boot, joint_boot = [], [], []
    for _ in range(BOOTSTRAP_REPS):
        indices = rng.integers(0, deficits.shape[0], size=deficits.shape[0])
        deficit_cycle = sustained_crossing_cycle(deficits[indices].mean(axis=0), cycles, GLOBAL_DEFICIT_THRESHOLD)
        rate_cycle = sustained_crossing_cycle(centered_rolling_mean_1d(rates[indices].mean(axis=0)), cycles, GLOBAL_RATE_THRESHOLD)
        deficit_boot.append(deficit_cycle)
        rate_boot.append(rate_cycle)
        joint_boot.append(max(deficit_cycle, rate_cycle))
    deficit_ci = percentile_interval(deficit_boot)
    rate_ci = percentile_interval(rate_boot)
    joint_ci = percentile_interval(joint_boot)
    global_transient_rows.append({
        'Nx': record['Nx'], 'Ny': record['Ny'],
        'deficit_threshold': GLOBAL_DEFICIT_THRESHOLD,
        'rate_threshold_per_cycle': GLOBAL_RATE_THRESHOLD,
        'deficit_crossing_cycle': deficit_point,
        'deficit_crossing_cycle_ci95_low': deficit_ci[0],
        'deficit_crossing_cycle_ci95_high': deficit_ci[1],
        'rate_crossing_cycle': rate_point,
        'rate_crossing_cycle_ci95_low': rate_ci[0],
        'rate_crossing_cycle_ci95_high': rate_ci[1],
        'operational_Tconv_cycle': joint_point,
        'operational_Tconv_cycle_ci95_low': joint_ci[0],
        'operational_Tconv_cycle_ci95_high': joint_ci[1],
        'bootstrap_repetitions': BOOTSTRAP_REPS,
    })
global_transient_df = pd.DataFrame(global_transient_rows)

wall_fit_data = spatial_covariance_region_df[
    (spatial_covariance_region_df['metric'] == 'reduced_covariance_temporal_rate')
    & (spatial_covariance_region_df['region'] == 'active_wall_rows')
].copy()
wall_fit_data['time_midpoint'] = 0.5 * (wall_fit_data['cycle_start'] + wall_fit_data['cycle_end'])

def fit_wall_model(frame, model):
    time = frame['time_midpoint'].to_numpy(dtype=np.float64)
    log_rate = np.log(frame['mean_rms_normalized'].to_numpy(dtype=np.float64))
    predictor = np.log(time) if model == 'power_law' else time
    design = np.column_stack([np.ones(time.size), predictor])
    beta = np.linalg.lstsq(design, log_rate, rcond=None)[0]
    fitted = design @ beta
    residual_sum_squares = float(np.sum((log_rate - fitted) ** 2))
    total_sum_squares = float(np.sum((log_rate - log_rate.mean()) ** 2))
    n_obs, n_parameters = time.size, design.shape[1]
    aicc = n_obs * np.log(residual_sum_squares / n_obs) + 2 * n_parameters + 2 * n_parameters * (n_parameters + 1) / (n_obs - n_parameters - 1)
    return {
        'model': model, 'intercept': beta[0], 'slope': beta[1],
        'amplitude': float(np.exp(beta[0])),
        'alpha': float(-beta[1]) if model == 'power_law' else np.nan,
        'tau_cycles': float(-1.0 / beta[1]) if model == 'exponential' else np.nan,
        'log_space_rss': residual_sum_squares,
        'log_space_r_squared': 1.0 - residual_sum_squares / total_sum_squares,
        'aicc': float(aicc),
    }

power_fit = fit_wall_model(wall_fit_data, 'power_law')
exponential_fit = fit_wall_model(wall_fit_data, 'exponential')
wall_sample_rates = spatial_sample_region_df[
    (spatial_sample_region_df['metric'] == 'reduced_covariance_temporal_rate')
    & (spatial_sample_region_df['region'] == 'active_wall_rows')
].copy()
power_alpha_boot, exponential_tau_boot, delta_aicc_boot = [], [], []
for _ in range(BOOTSTRAP_REPS):
    boot_rows = []
    for ny in SPATIAL_PILOT_NYS:
        geometry_rows = wall_sample_rates[wall_sample_rates['Ny'] == ny]
        sample_ids = np.sort(geometry_rows['sample_index'].unique())
        sampled_ids = rng.choice(sample_ids, size=sample_ids.size, replace=True)
        for cycle_start, cycle_end in zip(SPATIAL_PILOT_CYCLES[:-1], SPATIAL_PILOT_CYCLES[1:]):
            interval_rows = geometry_rows[(geometry_rows['cycle_start'] == cycle_start) & (geometry_rows['cycle_end'] == cycle_end)].set_index('sample_index')
            boot_rows.append({
                'Ny': ny, 'cycle_start': cycle_start, 'cycle_end': cycle_end,
                'time_midpoint': 0.5 * (cycle_start + cycle_end),
                'mean_rms_normalized': interval_rows.loc[sampled_ids, 'sample_region_mean_rms_normalized'].mean(),
            })
    boot_frame = pd.DataFrame(boot_rows)
    boot_power = fit_wall_model(boot_frame, 'power_law')
    boot_exponential = fit_wall_model(boot_frame, 'exponential')
    power_alpha_boot.append(boot_power['alpha'])
    exponential_tau_boot.append(boot_exponential['tau_cycles'])
    delta_aicc_boot.append(boot_exponential['aicc'] - boot_power['aicc'])
power_alpha_ci = percentile_interval(power_alpha_boot)
exponential_tau_ci = percentile_interval(exponential_tau_boot)
delta_aicc_ci = percentile_interval(delta_aicc_boot)
power_fit.update({'parameter_ci95_low': power_alpha_ci[0], 'parameter_ci95_high': power_alpha_ci[1]})
exponential_fit.update({'parameter_ci95_low': exponential_tau_ci[0], 'parameter_ci95_high': exponential_tau_ci[1]})
wall_decay_fit_df = pd.DataFrame([power_fit, exponential_fit])
wall_decay_fit_df['delta_aicc_relative_to_power_law'] = wall_decay_fit_df['aicc'] - power_fit['aicc']
wall_decay_fit_df['bootstrap_repetitions'] = BOOTSTRAP_REPS
wall_decay_fit_df['bootstrap_delta_aicc_exp_minus_power_ci95_low'] = delta_aicc_ci[0]
wall_decay_fit_df['bootstrap_delta_aicc_exp_minus_power_ci95_high'] = delta_aicc_ci[1]
wall_decay_fit_df['bootstrap_fraction_power_law_lower_aicc'] = np.mean(np.asarray(delta_aicc_boot) > 0.0)

transient_png_path = FIGURE_ROOT / 'quantitative_transient_timescale_and_wall_decay.png'
transient_pdf_path = FIGURE_ROOT / 'quantitative_transient_timescale_and_wall_decay.pdf'
transient_table_path = TABLE_ROOT / 'quantitative_global_transient_timescale.csv'
wall_decay_fit_table_path = TABLE_ROOT / 'wall_rate_decay_model_comparison.csv'
global_transient_df.to_csv(transient_table_path, index=False)
wall_decay_fit_df.to_csv(wall_decay_fit_table_path, index=False)

with plt.rc_context(style):
    fig, axes = plt.subplots(1, 2, figsize=(6.75, 2.8), constrained_layout=True)
    positions = np.arange(len(global_transient_df))
    for offset, column, low_column, high_column, label, marker, color in [
        (-0.08, 'deficit_crossing_cycle', 'deficit_crossing_cycle_ci95_low', 'deficit_crossing_cycle_ci95_high', 'deficit below 1e-3', 'o', '#0072B2'),
        (0.08, 'rate_crossing_cycle', 'rate_crossing_cycle_ci95_low', 'rate_crossing_cycle_ci95_high', 'rate below 1e-4/cycle', 's', '#D55E00'),
    ]:
        values = global_transient_df[column].to_numpy()
        lower = values - global_transient_df[low_column].to_numpy()
        upper = global_transient_df[high_column].to_numpy() - values
        axes[0].errorbar(positions + offset, values, yerr=np.vstack([lower, upper]), fmt=marker, color=color, capsize=3, label=label)
    axes[0].set_xticks(positions, [f"{int(nx)}x{int(ny)}" for nx, ny in zip(global_transient_df['Nx'], global_transient_df['Ny'])])
    axes[0].set_xlabel('system size')
    axes[0].set_ylabel('first sustained cycle')
    axes[0].set_title('(a) Operational global transient')
    axes[0].legend(frameon=False, loc='best')
    axes[0].grid(True, axis='y', alpha=0.22, linewidth=0.4)

    for color, ny, marker in zip(('#0072B2', '#D55E00'), SPATIAL_PILOT_NYS, ('o', 's')):
        group = wall_fit_data[wall_fit_data['Ny'] == ny]
        axes[1].errorbar(group['time_midpoint'], group['mean_rms_normalized'], yerr=group['sem_rms_normalized'], fmt=marker, color=color, capsize=2.5, label=f'16x{ny} data')
    fit_times = np.geomspace(wall_fit_data['time_midpoint'].min(), wall_fit_data['time_midpoint'].max(), 200)
    power_prediction = power_fit['amplitude'] * fit_times ** (-power_fit['alpha'])
    exponential_prediction = exponential_fit['amplitude'] * np.exp(-fit_times / exponential_fit['tau_cycles'])
    axes[1].plot(fit_times, power_prediction, color='#009E73', linewidth=1.4, label=rf"power: alpha={power_fit['alpha']:.2f}")
    axes[1].plot(fit_times, exponential_prediction, color='0.35', linestyle='--', linewidth=1.2, label=rf"exp: tau={exponential_fit['tau_cycles']:.1f} cycles")
    axes[1].set_xscale('log')
    axes[1].set_yscale('log')
    axes[1].set_xlabel('cycle-interval midpoint')
    axes[1].set_ylabel('wall RMS covariance rate / cycle')
    axes[1].set_title('(b) Persistent wall relaxation')
    axes[1].legend(frameon=False, loc='best')
    axes[1].grid(True, which='both', alpha=0.22, linewidth=0.4)
    fig.savefig(transient_png_path, dpi=300, bbox_inches='tight')
    fig.savefig(transient_pdf_path, dpi=300, bbox_inches='tight')
    plt.show()

display(global_transient_df)
display(wall_decay_fit_df)
print(f'Saved PNG: {transient_png_path}')
print(f'Saved PDF: {transient_pdf_path}')

## Wall versus bulk purification and finite-size scaling

The saved maximally mixed entropy-contour campaigns provide cycle-by-cycle spatial resolution. We define the wall sector as the two exact domain-wall rows and the active bulk as topological-slab rows at least two rows from either wall; the hard-truncated exterior is excluded. The wall is tested against power-law and exponential decay on cycles 5–30. The bulk is modeled as a rapidly contracting intrinsic term plus the algebraic tail inherited from the wall, $s_{\mathrm{bulk}}(t)=A_b e^{-t/\tau_b}+B_b t^{-\alpha_{\mathrm{wall}}}$. Threshold convergence means that the regional entropy density remains below the stated tolerance at every later saved cycle.


In [ ]:
PURIFICATION_REGION_CASES = (
    {
        'label': '16x30', 'Nx': 16, 'Ny': 30, 'walls': (5, 11),
        'path': SMALL_SYSTEM_BUNDLE_ROOT / 'gpu_data/purification_entropy_contours_maxmix/campaigns/N16x30_C50_dwtrunc1/runs/nsh1_perfect_correction/entropy_contour_time_maxmix.npz',
    },
    {
        'label': '16x32', 'Nx': 16, 'Ny': 32, 'walls': (5, 11),
        'path': SMALL_SYSTEM_BUNDLE_ROOT / 'gpu_data/purification_entropy_contours_maxmix/campaigns/N16x32_C100_dwtrunc1/runs/nsh1_perfect_correction/entropy_contour_time_maxmix.npz',
    },
    {
        'label': '30x40', 'Nx': 30, 'Ny': 40, 'walls': (9, 21),
        'path': SMALL_SYSTEM_BUNDLE_ROOT / 'gpu_data/purification_entropy_contours_maxmix/campaigns/N30x40_C100_dwtrunc1_nsh1/runs/nsh1_perfect_correction/entropy_contour_time_maxmix.npz',
    },
)
WALL_FIT_WINDOW = (5, 30)
WALL_WINDOW_ONSETS = (3, 5, 8, 10)
REGIONAL_THRESHOLDS = (1e-2, 1e-3, 1e-4)
ENTROPY_LOG_FLOOR = 1e-12

def fit_regional_log_model(cycles, values, model, cycle_min, cycle_max):
    mask = (cycles >= cycle_min) & (cycles <= cycle_max) & (values > ENTROPY_LOG_FLOOR)
    time = cycles[mask].astype(np.float64)
    log_values = np.log(values[mask])
    predictor = np.log(time) if model == 'power_law' else time
    design = np.column_stack([np.ones(time.size), predictor])
    beta = np.linalg.lstsq(design, log_values, rcond=None)[0]
    fitted = design @ beta
    residual_sum_squares = float(np.sum((log_values - fitted) ** 2))
    total_sum_squares = float(np.sum((log_values - log_values.mean()) ** 2))
    n_obs, n_parameters = time.size, design.shape[1]
    aicc = n_obs * np.log(residual_sum_squares / n_obs) + 2 * n_parameters + 2 * n_parameters * (n_parameters + 1) / (n_obs - n_parameters - 1)
    return {
        'model': model, 'amplitude': float(np.exp(beta[0])),
        'alpha': float(-beta[1]) if model == 'power_law' else np.nan,
        'tau_cycles': float(-1.0 / beta[1]) if model == 'exponential' else np.nan,
        'log_space_r_squared': 1.0 - residual_sum_squares / total_sum_squares,
        'aicc': float(aicc), 'fit_cycle_min': cycle_min, 'fit_cycle_max': cycle_max,
    }

purification_region_records = []
purification_region_rows = []
purification_fit_rows = []
purification_threshold_rows = []
purification_window_rows = []
purification_sources = []
for case in PURIFICATION_REGION_CASES:
    if not case['path'].exists():
        raise FileNotFoundError(case['path'])
    with np.load(case['path'], allow_pickle=False) as saved:
        cycles = saved['cycles'].astype(np.float64, copy=True)
        contour_bits = saved['entropy_contour_avg'].astype(np.float64, copy=True) / np.log(2.0)
        config = json.loads(str(saved['config_json'].item()))
    if contour_bits.shape != (cycles.size, case['Nx'], case['Ny']):
        raise ValueError(f"{case['label']}: entropy-contour shape mismatch")
    if config['protocol'] != 'perfect_correction' or config['init_mode'] != 'maxmix' or not config['dw_truncation'] or config['nshell'] != 1:
        raise ValueError(f"{case['label']}: unexpected purification protocol")
    x_coordinates = np.arange(case['Nx'])
    walls = np.asarray(case['walls'])
    distance_to_wall = np.min(np.abs(x_coordinates[:, None] - walls[None, :]), axis=1)
    active_slab = (x_coordinates >= walls[0]) & (x_coordinates <= walls[1])
    region_masks = {
        'wall_rows': np.isin(x_coordinates, walls),
        'active_bulk_rows': active_slab & (distance_to_wall >= 2),
    }
    region_curves = {}
    for region, region_mask in region_masks.items():
        curve = contour_bits[:, region_mask, :].mean(axis=(1, 2))
        region_curves[region] = curve
        for cycle, value in zip(cycles, curve):
            purification_region_rows.append({
                'Nx': case['Nx'], 'Ny': case['Ny'], 'geometry': case['label'],
                'cycle': int(cycle), 'region': region,
                'entropy_density_bits_per_cell': value,
            })
        for threshold in REGIONAL_THRESHOLDS:
            crossing = sustained_crossing_cycle(curve, cycles, threshold)
            purification_threshold_rows.append({
                'Nx': case['Nx'], 'Ny': case['Ny'], 'geometry': case['label'],
                'region': region, 'threshold_bits_per_cell': threshold,
                'sustained_crossing_cycle': crossing,
                'right_censored': bool(not np.isfinite(crossing)),
                'last_saved_cycle': int(cycles[-1]),
            })

    wall_curve = region_curves['wall_rows']
    bulk_curve = region_curves['active_bulk_rows']
    wall_power = fit_regional_log_model(cycles, wall_curve, 'power_law', *WALL_FIT_WINDOW)
    wall_exponential = fit_regional_log_model(cycles, wall_curve, 'exponential', *WALL_FIT_WINDOW)
    for fit in (wall_power, wall_exponential):
        purification_fit_rows.append({
            'Nx': case['Nx'], 'Ny': case['Ny'], 'geometry': case['label'],
            'region': 'wall_rows', **fit,
            'delta_aicc_relative_to_wall_power_law': fit['aicc'] - wall_power['aicc'],
        })
    for onset in WALL_WINDOW_ONSETS:
        onset_fit = fit_regional_log_model(cycles, wall_curve, 'power_law', onset, WALL_FIT_WINDOW[1])
        purification_window_rows.append({
            'Nx': case['Nx'], 'Ny': case['Ny'], 'geometry': case['label'],
            'fit_cycle_min': onset, 'fit_cycle_max': WALL_FIT_WINDOW[1],
            'wall_power_law_alpha': onset_fit['alpha'],
            'log_space_r_squared': onset_fit['log_space_r_squared'],
        })

    bulk_fit_mask = (cycles >= 1) & (cycles <= 30)
    def bulk_two_component(time, exponential_amplitude, tau_bulk, wall_tail_amplitude):
        return exponential_amplitude * np.exp(-time / tau_bulk) + wall_tail_amplitude * time ** (-wall_power['alpha'])
    bulk_parameters, _ = curve_fit(
        bulk_two_component, cycles[bulk_fit_mask], bulk_curve[bulk_fit_mask],
        p0=(2.0, 0.6, 0.02), bounds=((0.0, 0.05, 0.0), (10.0, 20.0, 10.0)), maxfev=100000,
    )
    bulk_prediction = bulk_two_component(cycles[bulk_fit_mask], *bulk_parameters)
    bulk_log_residual = np.log(bulk_curve[bulk_fit_mask]) - np.log(bulk_prediction)
    bulk_log_total = np.log(bulk_curve[bulk_fit_mask]) - np.log(bulk_curve[bulk_fit_mask]).mean()
    bulk_fit = {
        'model': 'exponential_plus_fixed_wall_power_law',
        'amplitude': bulk_parameters[0], 'alpha': wall_power['alpha'],
        'tau_cycles': bulk_parameters[1], 'wall_tail_amplitude': bulk_parameters[2],
        'log_space_r_squared': 1.0 - np.sum(bulk_log_residual ** 2) / np.sum(bulk_log_total ** 2),
        'aicc': np.nan, 'fit_cycle_min': 1, 'fit_cycle_max': 30,
        'delta_aicc_relative_to_wall_power_law': np.nan,
    }
    purification_fit_rows.append({
        'Nx': case['Nx'], 'Ny': case['Ny'], 'geometry': case['label'],
        'region': 'active_bulk_rows', **bulk_fit,
    })

    late_wall_exponential = None
    crossover_cycle_80pct = np.nan
    if cycles[-1] >= 100:
        late_wall_exponential = fit_regional_log_model(cycles, wall_curve, 'exponential', 20, 100)
        early_power_prediction = wall_power['amplitude'] * np.maximum(cycles, 1.0) ** (-wall_power['alpha'])
        ratio = wall_curve / early_power_prediction
        below = (ratio <= 0.8) & (cycles >= 30)
        sustained = np.logical_and.accumulate(below[::-1])[::-1]
        indices = np.flatnonzero(sustained)
        crossover_cycle_80pct = float(cycles[indices[0]]) if indices.size else np.nan
        purification_fit_rows.append({
            'Nx': case['Nx'], 'Ny': case['Ny'], 'geometry': case['label'],
            'region': 'wall_rows_late_finite_size_tail', **late_wall_exponential,
            'delta_aicc_relative_to_wall_power_law': np.nan,
        })

    wall_threshold = next(row for row in purification_threshold_rows if row['geometry'] == case['label'] and row['region'] == 'wall_rows' and row['threshold_bits_per_cell'] == 1e-2)
    bulk_threshold = next(row for row in purification_threshold_rows if row['geometry'] == case['label'] and row['region'] == 'active_bulk_rows' and row['threshold_bits_per_cell'] == 1e-2)
    window_alpha = [row['wall_power_law_alpha'] for row in purification_window_rows if row['geometry'] == case['label']]
    purification_region_records.append({
        **case, 'cycles': cycles, 'wall_curve': wall_curve, 'bulk_curve': bulk_curve,
        'wall_power': wall_power, 'bulk_parameters': bulk_parameters,
        'wall_alpha_window_min': min(window_alpha), 'wall_alpha_window_max': max(window_alpha),
        'wall_T_1e2': wall_threshold['sustained_crossing_cycle'],
        'bulk_T_1e2': bulk_threshold['sustained_crossing_cycle'],
        'late_wall_tau': late_wall_exponential['tau_cycles'] if late_wall_exponential else np.nan,
        'crossover_cycle_80pct': crossover_cycle_80pct,
    })
    purification_sources.append(case['path'])

purification_region_df = pd.DataFrame(purification_region_rows)
purification_fit_df = pd.DataFrame(purification_fit_rows)
purification_threshold_df = pd.DataFrame(purification_threshold_rows)
purification_window_df = pd.DataFrame(purification_window_rows)
purification_scaling_df = pd.DataFrame([{
    'Nx': record['Nx'], 'Ny': record['Ny'], 'geometry': record['label'],
    'wall_power_law_alpha_cycles_5_30': record['wall_power']['alpha'],
    'wall_alpha_window_sensitivity_min': record['wall_alpha_window_min'],
    'wall_alpha_window_sensitivity_max': record['wall_alpha_window_max'],
    'wall_exponential_minus_power_delta_aicc_cycles_5_30': next(row['delta_aicc_relative_to_wall_power_law'] for row in purification_fit_rows if row['geometry'] == record['label'] and row['region'] == 'wall_rows' and row['model'] == 'exponential'),
    'intrinsic_bulk_tau_cycles': record['bulk_parameters'][1],
    'wall_T_entropy_density_le_1e2': record['wall_T_1e2'],
    'bulk_T_entropy_density_le_1e2': record['bulk_T_1e2'],
    'late_wall_finite_size_tau_cycles_fit_20_100': record['late_wall_tau'],
    'wall_finite_size_crossover_cycle_ratio_le_0p8': record['crossover_cycle_80pct'],
    'last_saved_cycle': int(record['cycles'][-1]),
} for record in purification_region_records])

wall_bulk_png_path = FIGURE_ROOT / 'wall_bulk_purification_convergence_and_scaling.png'
wall_bulk_pdf_path = FIGURE_ROOT / 'wall_bulk_purification_convergence_and_scaling.pdf'
wall_bulk_region_table_path = TABLE_ROOT / 'wall_bulk_entropy_region_curves.csv'
wall_bulk_fit_table_path = TABLE_ROOT / 'wall_bulk_entropy_fit_summary.csv'
wall_bulk_threshold_table_path = TABLE_ROOT / 'wall_bulk_entropy_convergence_thresholds.csv'
wall_bulk_window_table_path = TABLE_ROOT / 'wall_power_law_window_sensitivity.csv'
wall_bulk_scaling_table_path = TABLE_ROOT / 'wall_bulk_purification_scaling_summary.csv'
purification_region_df.to_csv(wall_bulk_region_table_path, index=False)
purification_fit_df.to_csv(wall_bulk_fit_table_path, index=False)
purification_threshold_df.to_csv(wall_bulk_threshold_table_path, index=False)
purification_window_df.to_csv(wall_bulk_window_table_path, index=False)
purification_scaling_df.to_csv(wall_bulk_scaling_table_path, index=False)

regional_colors = ('#0072B2', '#D55E00', '#009E73')
with plt.rc_context(style):
    fig, axes = plt.subplots(2, 2, figsize=(6.75, 5.0), constrained_layout=True)
    for color, record in zip(regional_colors, purification_region_records):
        cycles = record['cycles']
        wall_curve = record['wall_curve']
        bulk_curve = record['bulk_curve']
        axes[0, 0].plot(cycles[1:], wall_curve[1:], color=color, linewidth=1.1, label=record['label'])
        fit_cycles = np.arange(WALL_FIT_WINDOW[0], WALL_FIT_WINDOW[1] + 1, dtype=np.float64)
        axes[0, 0].plot(fit_cycles, record['wall_power']['amplitude'] * fit_cycles ** (-record['wall_power']['alpha']), color=color, linestyle='--', linewidth=0.9)
        axes[0, 1].plot(cycles[1:], bulk_curve[1:], color=color, linewidth=1.1, label=record['label'])
        bulk_fit_cycles = np.arange(1, 31, dtype=np.float64)
        bulk_amplitude, bulk_tau, bulk_tail = record['bulk_parameters']
        bulk_prediction = bulk_amplitude * np.exp(-bulk_fit_cycles / bulk_tau) + bulk_tail * bulk_fit_cycles ** (-record['wall_power']['alpha'])
        axes[0, 1].plot(bulk_fit_cycles, bulk_prediction, color=color, linestyle='--', linewidth=0.9)
    axes[0, 0].set_xscale('log')
    axes[0, 0].set_yscale('log')
    axes[0, 0].set_xlabel('cycle')
    axes[0, 0].set_ylabel('wall entropy density [bits/cell]')
    axes[0, 0].set_title('(a) Algebraic wall purification')
    axes[0, 0].legend(frameon=False, loc='best')
    axes[0, 1].set_yscale('log')
    axes[0, 1].set_xlabel('cycle')
    axes[0, 1].set_ylabel('active-bulk entropy density [bits/cell]')
    axes[0, 1].set_title('(b) Fast bulk contraction plus wall tail')
    axes[0, 1].legend(frameon=False, loc='best')

    positions = np.arange(len(purification_region_records))
    axes[1, 0].plot(positions - 0.08, [record['bulk_T_1e2'] for record in purification_region_records], 's', color='#0072B2', label='active bulk')
    axes[1, 0].plot(positions + 0.08, [record['wall_T_1e2'] for record in purification_region_records], 'o', color='#D55E00', label='wall')
    axes[1, 0].set_xticks(positions, [record['label'] for record in purification_region_records])
    axes[1, 0].set_xlabel('system size')
    axes[1, 0].set_ylabel('sustained crossing cycle')
    axes[1, 0].set_title('(c) Convergence at 0.01 bits/cell')
    axes[1, 0].legend(frameon=False, loc='best')

    alpha_values = np.asarray([record['wall_power']['alpha'] for record in purification_region_records])
    alpha_lower = alpha_values - np.asarray([record['wall_alpha_window_min'] for record in purification_region_records])
    alpha_upper = np.asarray([record['wall_alpha_window_max'] for record in purification_region_records]) - alpha_values
    axes[1, 1].errorbar([record['Ny'] for record in purification_region_records], alpha_values, yerr=np.vstack([alpha_lower, alpha_upper]), fmt='o', color='#009E73', capsize=3)
    for record in purification_region_records:
        axes[1, 1].annotate(record['label'], (record['Ny'], record['wall_power']['alpha']), xytext=(4, 3), textcoords='offset points')
    axes[1, 1].set_xlabel('$N_y$')
    axes[1, 1].set_ylabel('wall exponent alpha')
    axes[1, 1].set_title('(d) Weak exponent drift')
    for axis in axes.flat:
        axis.grid(True, which='both', alpha=0.22, linewidth=0.4)
    fig.savefig(wall_bulk_png_path, dpi=300, bbox_inches='tight')
    fig.savefig(wall_bulk_pdf_path, dpi=300, bbox_inches='tight')
    plt.show()

display(purification_scaling_df)
display(purification_threshold_df)
print(f'Saved PNG: {wall_bulk_png_path}')
print(f'Saved PDF: {wall_bulk_pdf_path}')

## Born-rule total-charge trace

For trajectory $s$, spatially summing the saved cell charges gives $n_s(t)=\operatorname{tr}C_s(t)=\langle\hat N_F\rangle_s$. Because these 100 trajectories were sampled with Born-rule outcomes and `postselect=False`, their arithmetic mean estimates the Born-rule ensemble mean. There are $V=2N_xN_y$ single-particle modes and half filling is $V/2=N_xN_y$. The left panel shows $[\mathbb E_{\rm Born}(n_s)-V/2]/V$. The right panel shows the extensive trajectory population variance $S^{-1}\sum_s[n_s(t)-\bar n(t)]^2$ divided by $V$. Thick curves use the same centered 5-cycle rolling average (`min_periods=1`) as the prior May 6 analysis; faint curves show the unsmoothed values. This is distinct from the within-trajectory quantum variance $\operatorname{tr}(C-C^2)$ above.

In [ ]:
trace_png_path = FIGURE_ROOT / 'born_rule_total_charge_trace_mean_and_variance.png'
trace_pdf_path = FIGURE_ROOT / 'born_rule_total_charge_trace_mean_and_variance.pdf'

with plt.rc_context(style):
    fig, axes = plt.subplots(1, 2, figsize=(6.75, 2.65), constrained_layout=True)
    for color, record in zip(colors, records):
        cycles = record['cycles']
        centered_mean_trace = record['normalized_trace_offset_mean']
        rolling_centered_mean_trace = record['rolling_normalized_trace_offset_mean']
        rolling_trace_sem = record['rolling_normalized_trace_offset_sem']
        trace_variance = record['normalized_trace_population_variance']
        rolling_trace_variance = record['rolling_normalized_trace_population_variance']

        axes[0].plot(cycles, centered_mean_trace, color=color, alpha=0.20, linewidth=0.65)
        axes[0].plot(cycles, rolling_centered_mean_trace, color=color, linewidth=1.45, label=record['label'])
        axes[0].fill_between(
            cycles, rolling_centered_mean_trace - rolling_trace_sem, rolling_centered_mean_trace + rolling_trace_sem,
            color=color, alpha=0.14, linewidth=0,
        )
        axes[1].plot(cycles, trace_variance, color=color, alpha=0.20, linewidth=0.65)
        axes[1].plot(cycles, rolling_trace_variance, color=color, linewidth=1.45, label=record['label'])

    axes[0].axhline(0.0, color='0.35', linewidth=0.65, linestyle='--', zorder=0)
    axes[0].set_xlabel('cycle')
    axes[0].set_ylabel(r'$[\mathbb{E}_{\rm Born}(\operatorname{tr}C)-N_xN_y]/(2N_xN_y)$')
    axes[0].set_title(f'(a) Normalized mean trace offset, rolling $w={ROLLING_WINDOW}$')
    axes[1].set_xlabel('cycle')
    axes[1].set_ylabel(r'$\operatorname{Var}_{\rm traj}(\operatorname{tr}C)/(2N_xN_y)$')
    axes[1].set_title(f'(b) Normalized trace variance, rolling $w={ROLLING_WINDOW}$')

    for ax in axes:
        ax.grid(True, alpha=0.22, linewidth=0.4)
        ax.legend(frameon=False, loc='best')

    fig.savefig(trace_png_path, dpi=300, bbox_inches='tight')
    fig.savefig(trace_pdf_path, dpi=300, bbox_inches='tight')
    plt.show()

print(f'Saved PNG: {trace_png_path}')
print(f'Saved PDF: {trace_pdf_path}')

## Boundary and bulk localization of charge statistics

This section repairs the earlier wall/bulk estimator by measuring deviations from the half-filled cell occupancy, not the occupancy itself. Profiles average over $y$ and the late-time half of each run (cycles $N_y+1$ through $2N_y$). Panel (a) is the absolute bias of the ensemble-mean expected occupancy from one particle per cell. Panel (b) is the trajectory-to-trajectory variance of that expected occupancy. Panel (c) is the distinct within-trajectory quantum variance saved by the Gaussian observer. The shaded interval is the active slab $5\le x\le15$ and dotted lines mark its two domain walls.

In [ ]:
localization_png_path = FIGURE_ROOT / 'late_time_boundary_bulk_charge_profiles.png'
localization_pdf_path = FIGURE_ROOT / 'late_time_boundary_bulk_charge_profiles.pdf'
profile_specs = [
    ('occupancy_bias_x', r'$\overline{|\mathbb{E}_{\rm Born}[n(x,y,t)]-1|}_{t,y}$', '(a) Ensemble-mean occupancy bias'),
    ('trajectory_variance_x', r'$\overline{\operatorname{Var}_{\rm traj}[n(x,y,t)]}_{t,y}$', '(b) Between-trajectory variance'),
    ('quantum_local_variance_x', r'$\overline{\mathbb{E}_{\rm traj}[\operatorname{Var}(\hat N_{x,y})]}_{t,y}$', '(c) Within-trajectory quantum variance'),
]

with plt.rc_context(style):
    fig, axes = plt.subplots(1, 3, figsize=(10.125, 2.7), constrained_layout=True, sharex=True)
    for ax, (profile_key, ylabel, title) in zip(axes, profile_specs):
        ax.axvspan(5, 15, color='0.5', alpha=0.08, linewidth=0, zorder=0)
        ax.axvline(5, color='0.35', linestyle=':', linewidth=0.75, zorder=1)
        ax.axvline(15, color='0.35', linestyle=':', linewidth=0.75, zorder=1)
        for color, record in zip(colors, records):
            ax.plot(np.arange(record['Nx']), record[profile_key], color=color, marker='o', markersize=2.2, label=record['label'])
        ax.set_xlabel('$x$')
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.set_yscale('log')
        ax.set_xticks((0, 5, 10, 15, 19))
        ax.grid(True, alpha=0.22, linewidth=0.4)
    axes[0].legend(frameon=False, loc='best')
    fig.savefig(localization_png_path, dpi=300, bbox_inches='tight')
    fig.savefig(localization_pdf_path, dpi=300, bbox_inches='tight')
    plt.show()

print(f'Saved PNG: {localization_png_path}')
print(f'Saved PDF: {localization_pdf_path}')

## Raw summary and diagnostics

This final section saves the plotted temporal and spatial summaries, Frobenius convergence thresholds, adjacent-cycle product-residual statistics, the reduced-covariance spatial pilot, every reconstructed trajectory trace, and the localization/capture statistics as tidy CSV files. It also writes a provenance manifest and prints the finite-value checks used by the figures.

In [ ]:
def first_sustained_cycle(values, cycles, threshold):
    values = np.asarray(values, dtype=np.float64)
    suffix_max = np.maximum.accumulate(values[::-1])[::-1]
    matches = np.flatnonzero(suffix_max <= threshold)
    return int(cycles[matches[0]]) if matches.size else np.nan

rows = []
frob_rows = []
frob_convergence_rows = []
product_rows = []
product_summary_rows = []
trace_summary_rows = []
trajectory_trace_rows = []
spatial_profile_rows = []
localization_rows = []
diagnostics = []
for record in records:
    for cycle, mean, sem, derivative_mean, derivative_sem in zip(
        record['cycles'],
        record['charge_variance_mean'],
        record['charge_variance_sem'],
        record['absolute_derivative_mean'],
        record['absolute_derivative_sem'],
    ):
        rows.append({
            'Nx': record['Nx'],
            'Ny': record['Ny'],
            'cycle': int(cycle),
            'charge_variance_mean': mean,
            'charge_variance_sem': sem,
            'absolute_charge_variance_cycle_derivative_mean': derivative_mean,
            'absolute_charge_variance_cycle_derivative_sem': derivative_sem,
        })
    for cycle, norm_mean, norm_sem, deficit_mean, rate_mean, rate_sem, rolling_rate in zip(
        record['cycles'],
        record['frob_norm_mean'],
        record['frob_norm_sem'],
        record['frob_normalized_deficit_mean'],
        record['frob_absolute_derivative_mean'],
        record['frob_absolute_derivative_sem'],
        record['rolling_frob_absolute_derivative_mean'],
    ):
        frob_rows.append({
            'Nx': record['Nx'],
            'Ny': record['Ny'],
            'cycle': int(cycle),
            'pure_state_frobenius_ceiling': record['frob_ceiling'],
            'pure_state_frobenius_ceiling_per_2NxNy': record['frob_ceiling_per_mode'],
            'pure_state_frobenius_ceiling_rms_normalized': record['frob_ceiling_rms'],
            'frobenius_norm_mean': norm_mean,
            'frobenius_norm_sem': norm_sem,
            'frobenius_norm_mean_per_2NxNy': norm_mean / (2.0 * record['Nx'] * record['Ny']),
            'frobenius_norm_sem_per_2NxNy': norm_sem / (2.0 * record['Nx'] * record['Ny']),
            'frobenius_norm_mean_rms_normalized': norm_mean / np.sqrt(2.0 * record['Nx'] * record['Ny']),
            'frobenius_norm_sem_rms_normalized': norm_sem / np.sqrt(2.0 * record['Nx'] * record['Ny']),
            'normalized_frobenius_deficit_mean': deficit_mean,
            'absolute_frobenius_norm_cycle_derivative_mean': rate_mean,
            'absolute_frobenius_norm_cycle_derivative_sem': rate_sem,
            'absolute_frobenius_norm_cycle_derivative_mean_per_2NxNy': rate_mean / (2.0 * record['Nx'] * record['Ny']),
            'absolute_frobenius_norm_cycle_derivative_sem_per_2NxNy': rate_sem / (2.0 * record['Nx'] * record['Ny']),
            'absolute_frobenius_norm_cycle_derivative_mean_rms_normalized': rate_mean / np.sqrt(2.0 * record['Nx'] * record['Ny']),
            'absolute_frobenius_norm_cycle_derivative_sem_rms_normalized': rate_sem / np.sqrt(2.0 * record['Nx'] * record['Ny']),
            f'absolute_frobenius_norm_cycle_derivative_mean_rms_normalized_rolling_w{ROLLING_WINDOW}': rolling_rate / np.sqrt(2.0 * record['Nx'] * record['Ny']),
            f'absolute_frobenius_norm_cycle_derivative_mean_rolling_w{ROLLING_WINDOW}': rolling_rate,
            f'normalized_absolute_frobenius_rate_rolling_w{ROLLING_WINDOW}': rolling_rate / record['frob_ceiling'],
        })
    normalized_rolling_rate = record['rolling_frob_absolute_derivative_mean'] / record['frob_ceiling']
    tail_count = max(1, record['cycles'].size // 5)
    frob_convergence_rows.append({
        'Nx': record['Nx'],
        'Ny': record['Ny'],
        'pure_state_frobenius_ceiling': record['frob_ceiling'],
        'initial_frobenius_norm_mean': record['frob_norm_mean'][0],
        'final_frobenius_norm_mean': record['frob_norm_mean'][-1],
        'final_normalized_frobenius_deficit': record['frob_normalized_deficit_mean'][-1],
        'tail_20_percent_normalized_frobenius_deficit_mean': record['frob_normalized_deficit_mean'][-tail_count:].mean(),
        f'final_normalized_absolute_frobenius_rate_rolling_w{ROLLING_WINDOW}': normalized_rolling_rate[-1],
        f'tail_20_percent_normalized_absolute_frobenius_rate_rolling_w{ROLLING_WINDOW}_mean': normalized_rolling_rate[-tail_count:].mean(),
        'first_sustained_cycle_deficit_le_1e-2': first_sustained_cycle(record['frob_normalized_deficit_mean'], record['cycles'], 1e-2),
        'first_sustained_cycle_deficit_le_1e-3': first_sustained_cycle(record['frob_normalized_deficit_mean'], record['cycles'], 1e-3),
        'first_sustained_cycle_deficit_le_1e-4': first_sustained_cycle(record['frob_normalized_deficit_mean'], record['cycles'], 1e-4),
        f'first_sustained_cycle_normalized_rate_rolling_w{ROLLING_WINDOW}_le_1e-3': first_sustained_cycle(normalized_rolling_rate, record['cycles'], 1e-3),
        f'first_sustained_cycle_normalized_rate_rolling_w{ROLLING_WINDOW}_le_1e-4': first_sustained_cycle(normalized_rolling_rate, record['cycles'], 1e-4),
    })
    for cycle, trace_mean, trace_sem, trace_variance, normalized_mean, normalized_sem, normalized_variance, rolling_mean, rolling_sem, rolling_variance in zip(
        record['cycles'],
        record['total_charge_trace_mean'],
        record['total_charge_trace_sem'],
        record['total_charge_trace_population_variance'],
        record['normalized_trace_offset_mean'],
        record['normalized_trace_offset_sem'],
        record['normalized_trace_population_variance'],
        record['rolling_normalized_trace_offset_mean'],
        record['rolling_normalized_trace_offset_sem'],
        record['rolling_normalized_trace_population_variance'],
    ):
        trace_summary_rows.append({
            'Nx': record['Nx'],
            'Ny': record['Ny'],
            'cycle': int(cycle),
            'born_rule_mean_trace': trace_mean,
            'born_rule_mean_trace_offset_from_half_filling': trace_mean - record['half_filling_trace'],
            'normalized_born_rule_mean_trace_offset': normalized_mean,
            'born_rule_mean_trace_sem': trace_sem,
            'normalized_born_rule_mean_trace_sem': normalized_sem,
            'trajectory_trace_population_variance': trace_variance,
            'normalized_trajectory_trace_population_variance': normalized_variance,
            f'normalized_born_rule_mean_trace_offset_rolling_w{ROLLING_WINDOW}': rolling_mean,
            f'normalized_born_rule_mean_trace_sem_rolling_w{ROLLING_WINDOW}': rolling_sem,
            f'normalized_trajectory_trace_population_variance_rolling_w{ROLLING_WINDOW}': rolling_variance,
        })
    for sample_index, trajectory_trace in zip(record['sample_indices'], record['total_charge_trace']):
        for cycle, trace_value in zip(record['cycles'], trajectory_trace):
            trajectory_trace_rows.append({
                'Nx': record['Nx'],
                'Ny': record['Ny'],
                'sample_index': int(sample_index),
                'cycle': int(cycle),
                'total_charge_trace': trace_value,
                'total_charge_trace_fraction_of_capacity': trace_value / (2.0 * float(record['Nx'] * record['Ny'])),
                'normalized_trace_offset_from_half_filling': (trace_value - record['half_filling_trace']) / (2.0 * float(record['Nx'] * record['Ny'])),
            })
    for x, distance, occupancy_bias, trajectory_variance, quantum_variance in zip(
        np.arange(record['Nx']),
        record['distance_to_wall'],
        record['occupancy_bias_x'],
        record['trajectory_variance_x'],
        record['quantum_local_variance_x'],
    ):
        spatial_profile_rows.append({
            'Nx': record['Nx'],
            'Ny': record['Ny'],
            'late_cycle_min': record['late_cycle_min'],
            'late_cycle_max': record['late_cycle_max'],
            'x': int(x),
            'distance_to_nearest_wall': int(distance),
            'ensemble_mean_occupancy_abs_bias': occupancy_bias,
            'trajectory_occupancy_population_variance': trajectory_variance,
            'mean_within_trajectory_quantum_variance': quantum_variance,
        })
    x_coords = np.arange(record['Nx'])
    distance = record['distance_to_wall']
    active_slab_mask = (x_coords >= 5) & (x_coords <= 15)
    active_wall_mask = active_slab_mask & (distance <= 1)
    active_bulk_mask = active_slab_mask & (distance >= 2)
    exterior_mask = ~active_slab_mask
    for metric, profile in (
        ('ensemble_mean_occupancy_abs_bias', record['occupancy_bias_x']),
        ('trajectory_occupancy_population_variance', record['trajectory_variance_x']),
        ('mean_within_trajectory_quantum_variance', record['quantum_local_variance_x']),
    ):
        total_weight = float(np.sum(profile))
        capture = {radius: float(np.sum(profile[distance <= radius]) / total_weight) for radius in range(6)}
        radius_90 = next(radius for radius in range(6) if capture[radius] >= 0.90)
        active_wall_mean = float(np.mean(profile[active_wall_mask]))
        active_bulk_mean = float(np.mean(profile[active_bulk_mask]))
        localization_rows.append({
            'Nx': record['Nx'],
            'Ny': record['Ny'],
            'late_cycle_min': record['late_cycle_min'],
            'late_cycle_max': record['late_cycle_max'],
            'metric': metric,
            'total_x_profile_weight': total_weight,
            'capture_fraction_on_wall_rows_r0': capture[0],
            'capture_fraction_within_wall_distance_1': capture[1],
            'capture_fraction_within_wall_distance_2': capture[2],
            'wall_distance_for_90_percent_capture': int(radius_90),
            'capture_fraction_in_active_slab_x5_to_x15': float(np.sum(profile[active_slab_mask]) / total_weight),
            'active_wall_mean_x5_x6_x14_x15': active_wall_mean,
            'active_bulk_mean_x7_to_x13': active_bulk_mean,
            'active_wall_to_bulk_mean_ratio': active_wall_mean / active_bulk_mean,
            'exterior_mean': float(np.mean(profile[exterior_mask])),
        })
    diagnostics.append({
        'geometry': f"N{record['Nx']}x{record['Ny']}",
        'samples': record['charge_variance'].shape[0],
        'cycles': record['charge_variance'].shape[1],
        'finite_variance_values': int(np.isfinite(record['charge_variance']).sum()),
        'total_variance_values': int(record['charge_variance'].size),
        'first_cycle_mean': float(record['charge_variance_mean'][0]),
        'last_cycle_mean': float(record['charge_variance_mean'][-1]),
        'minimum_mean': float(record['charge_variance_mean'].min()),
        'maximum_mean': float(record['charge_variance_mean'].max()),
        'minimum_absolute_derivative_mean': float(record['absolute_derivative_mean'].min()),
        'maximum_absolute_derivative_mean': float(record['absolute_derivative_mean'].max()),
        'pure_state_frobenius_ceiling': float(record['frob_ceiling']),
        'initial_frobenius_norm_mean': float(record['frob_norm_mean'][0]),
        'final_frobenius_norm_mean': float(record['frob_norm_mean'][-1]),
        'final_normalized_frobenius_deficit': float(record['frob_normalized_deficit_mean'][-1]),
        'final_absolute_frobenius_rate_mean': float(record['frob_absolute_derivative_mean'][-1]),
        'finite_trace_values': int(np.isfinite(record['total_charge_trace']).sum()),
        'total_trace_values': int(record['total_charge_trace'].size),
        'final_born_rule_mean_trace': float(record['total_charge_trace_mean'][-1]),
        'final_mean_trace_offset_from_half_filling': float(record['total_charge_trace_mean'][-1] - record['half_filling_trace']),
        'final_trajectory_trace_population_variance': float(record['total_charge_trace_population_variance'][-1]),
        'final_normalized_mean_trace_offset': float((record['total_charge_trace_mean'][-1] - record['half_filling_trace']) / (2.0 * record['Nx'] * record['Ny'])),
        'final_normalized_trajectory_trace_population_variance': float(record['total_charge_trace_population_variance'][-1] / (2.0 * record['Nx'] * record['Ny'])),
        'final_rolling_normalized_mean_trace_offset': float(record['rolling_normalized_trace_offset_mean'][-1]),
        'final_rolling_normalized_trajectory_trace_population_variance': float(record['rolling_normalized_trace_population_variance'][-1]),
    })

for record in product_records:
    for cycle, residual_mean, residual_sem, rolling_residual, rate_mean, rate_sem, rolling_rate in zip(
        record['cycles'],
        record['product_residual_mean'],
        record['product_residual_sem'],
        record['rolling_product_residual_mean'],
        record['product_absolute_derivative_mean'],
        record['product_absolute_derivative_sem'],
        record['rolling_product_absolute_derivative_mean'],
    ):
        product_rows.append({
            'Nx': record['Nx'],
            'Ny': record['Ny'],
            'cycle_transition_ending_at': int(cycle),
            'product_residual_frobenius_mean': residual_mean,
            'product_residual_frobenius_sem': residual_sem,
            f'product_residual_frobenius_mean_rolling_w{ROLLING_WINDOW}': rolling_residual,
            'product_residual_normalized_by_sqrt_2NxNy': residual_mean / record['pure_frobenius_scale'],
            'absolute_product_residual_cycle_derivative_mean': rate_mean,
            'absolute_product_residual_cycle_derivative_sem': rate_sem,
            f'absolute_product_residual_cycle_derivative_mean_rolling_w{ROLLING_WINDOW}': rolling_rate,
            f'absolute_product_residual_rate_normalized_by_sqrt_2NxNy_rolling_w{ROLLING_WINDOW}': rolling_rate / record['pure_frobenius_scale'],
        })
    tail_count = max(1, record['cycles'].size // 5)
    product_summary_rows.append({
        'Nx': record['Nx'],
        'Ny': record['Ny'],
        'first_product_residual_mean': record['product_residual_mean'][0],
        'final_product_residual_mean': record['product_residual_mean'][-1],
        'tail_20_percent_product_residual_mean': record['product_residual_mean'][-tail_count:].mean(),
        'tail_20_percent_product_residual_mean_normalized_by_sqrt_2NxNy': record['product_residual_mean'][-tail_count:].mean() / record['pure_frobenius_scale'],
        'tail_to_first_product_residual_ratio': record['product_residual_mean'][-tail_count:].mean() / record['product_residual_mean'][0],
        'final_absolute_product_residual_cycle_derivative_mean': record['product_absolute_derivative_mean'][-1],
        'tail_20_percent_absolute_product_residual_cycle_derivative_mean': record['product_absolute_derivative_mean'][-tail_count:].mean(),
        'tail_20_percent_absolute_product_residual_rate_normalized_by_sqrt_2NxNy': record['product_absolute_derivative_mean'][-tail_count:].mean() / record['pure_frobenius_scale'],
    })

plot_data_df = pd.DataFrame(rows)
frob_df = pd.DataFrame(frob_rows)
frob_convergence_df = pd.DataFrame(frob_convergence_rows)
product_df = pd.DataFrame(product_rows)
product_summary_df = pd.DataFrame(product_summary_rows)
trace_summary_df = pd.DataFrame(trace_summary_rows)
trajectory_trace_df = pd.DataFrame(trajectory_trace_rows)
spatial_profile_df = pd.DataFrame(spatial_profile_rows)
localization_df = pd.DataFrame(localization_rows)
diagnostics_df = pd.DataFrame(diagnostics)
csv_path = TABLE_ROOT / 'total_charge_variance_and_cycle_derivative.csv'
frob_csv_path = TABLE_ROOT / 'frobenius_norm_and_cycle_derivative.csv'
frob_convergence_csv_path = TABLE_ROOT / 'frobenius_convergence_summary.csv'
product_csv_path = TABLE_ROOT / 'adjacent_cycle_covariance_product_residual_and_derivative.csv'
product_summary_csv_path = TABLE_ROOT / 'adjacent_cycle_covariance_product_residual_summary.csv'
trace_summary_csv_path = TABLE_ROOT / 'born_rule_total_charge_trace_summary.csv'
trajectory_trace_csv_path = TABLE_ROOT / 'born_rule_total_charge_trace_by_trajectory.csv'
spatial_profile_csv_path = TABLE_ROOT / 'late_time_charge_x_profiles.csv'
localization_csv_path = TABLE_ROOT / 'late_time_charge_localization_summary.csv'
spatial_covariance_profile_csv_path = TABLE_ROOT / 'spatial_reduced_covariance_convergence_profiles.csv'
spatial_covariance_region_csv_path = TABLE_ROOT / 'spatial_reduced_covariance_region_summary.csv'
spatial_covariance_rate_ratio_csv_path = TABLE_ROOT / 'spatial_reduced_covariance_wall_bulk_rate_ratios.csv'
manifest_path = ANALYSIS_ROOT / 'analysis_manifest.json'
plot_data_df.to_csv(csv_path, index=False)
frob_df.to_csv(frob_csv_path, index=False)
frob_convergence_df.to_csv(frob_convergence_csv_path, index=False)
product_df.to_csv(product_csv_path, index=False)
product_summary_df.to_csv(product_summary_csv_path, index=False)
trace_summary_df.to_csv(trace_summary_csv_path, index=False)
trajectory_trace_df.to_csv(trajectory_trace_csv_path, index=False)
spatial_profile_df.to_csv(spatial_profile_csv_path, index=False)
localization_df.to_csv(localization_csv_path, index=False)
spatial_covariance_profile_df.to_csv(spatial_covariance_profile_csv_path, index=False)
spatial_covariance_region_df.to_csv(spatial_covariance_region_csv_path, index=False)
rate_region_wide.to_csv(spatial_covariance_rate_ratio_csv_path, index=False)
manifest = {
    'analysis_name': 'total_charge_variance_cycle_derivative',
    'campaign_id': CAMPAIGN_ID,
    'pure_streaming_campaign_id': STREAMING_CAMPAIGN_ID,
    'protocol': 'perfect_correction',
    'observable': records[0]['formula'],
    'frobenius_norm_observable': 'plotted quantity ||G||_F/sqrt(2*Nx*Ny), with ||G||_F = sqrt(2*Nx*Ny - 4*Var(N_F)) reconstructed exactly trajectory by trajectory from Tr(C-C^2), C=(G+I)/2, and Hermitian G; pure-state ceiling is 1',
    'frobenius_norm_source_ensemble': 'init-maxmix purification campaign',
    'frobenius_norm_derivative': f'plotted after division by sqrt(2*Nx*Ny); trajectory mean of absolute numpy.gradient values with raw mean and SEM plus centered rolling mean w={ROLLING_WINDOW}',
    'frobenius_convergence_thresholds': 'first saved cycle after which the ensemble-mean normalized Frobenius deficit remains <= 1e-2, 1e-3, or 1e-4; analogous normalized rolling-rate thresholds use 1e-3 and 1e-4 per cycle',
    'quantitative_transient_definition': {
        'operational_Tconv': 'later of the first sustained normalized Frobenius-deficit crossing below 1e-3 and the first sustained centered-w5 RMS-normalized Frobenius-rate crossing below 1e-4 per cycle',
        'uncertainty': f'trajectory bootstrap with {BOOTSTRAP_REPS} repetitions and seed {BOOTSTRAP_SEED}; percentile 95% intervals',
        'wall_decay_models': 'common-amplitude log-space fits to six geometry-interval means at interval midpoints: A*t^(-alpha) versus A*exp(-t/tau); trajectory bootstrap resamples paired intervals within each geometry',
        'scope_caveat': 'global thresholds diagnose saturation of the norm-derived ensemble observable, not cessation of trajectorywise covariance motion; the spatial pilot supports motion through cycle 50 but contains no observations after cycle 50',
    },
    'wall_bulk_purification_analysis': {
        'observable': 'sample-averaged spatial entropy contour divided by ln(2), averaged over cells in each region',
        'wall_region': 'the two exact domain-wall x rows',
        'bulk_region': 'active topological-slab x rows at distance at least two from either wall; hard-truncated exterior excluded',
        'wall_fit': 'log-space power-law and exponential comparison on cycles 5-30, with power-law onset sensitivity over cycles 3, 5, 8, and 10 at fixed endpoint 30',
        'bulk_fit': 'A_b*exp(-t/tau_b)+B_b*t^(-alpha_wall) on cycles 1-30, with alpha_wall fixed from the corresponding wall fit',
        'convergence_definition': 'first cycle after which regional entropy density remains below the selected bits-per-cell threshold',
        'scaling_caveat': 'only N16x30 and N16x32 hold Nx nearly fixed, and their Ny range is too narrow for a dynamical scaling exponent; N30x40 changes both dimensions',
    },
    'adjacent_cycle_product_residual': {
        'observable': '||G_(t-1) G_t - I||_F',
        'source_column': 'frob_successive_delta',
        'exact_identity': 'for pure Hermitian covariance involutions, ||G_(t-1) G_t-I||_F = ||G_t-G_(t-1)||_F',
        'cycle_label': 'transition ending at the listed cycle; cycle 1 compares the initial covariance with the first completed cycle',
        'source_ensemble': 'init-default pure-state streaming campaign',
        'covariance_history_saved': False,
        'rolling_average': {'window': ROLLING_WINDOW, 'center': ROLLING_CENTER, 'min_periods': ROLLING_MIN_PERIODS},
        'scope_caveat': 'global trajectorywise motion is not spatially resolved; attributing it specifically to the walls requires a restricted-block observer or saved covariance matrices',
    },
    'spatial_reduced_covariance_pilot': {
        'source_campaign': 'colab_small_system_testing/pure_state_covariance_snapshots',
        'scope': 'separate sparse-cycle pure-state pilot; not the N20 production ensemble',
        'geometries': [[16, 30], [16, 40]],
        'samples': SPATIAL_PILOT_SAMPLES,
        'nshell': SPATIAL_PILOT_NSHELL,
        'snapshot_cycles': list(SPATIAL_PILOT_CYCLES),
        'walls': [4, 12],
        'dw_truncation': True,
        'reduction': 'G_x is the principal covariance submatrix retaining both orbitals at fixed x for every y (2*Ny modes), equivalent to tracing out all other x rows for a Gaussian state',
        'temporal_rate': '||G_x(t_b)-G_x(t_a)||_F / ((t_b-t_a)*sqrt(2*Ny))',
        'mixedness': '||G_x(t)^2-I_x||_F/sqrt(2*Ny)',
        'normalization': 'RMS spectral normalization by square root of retained covariance dimension',
        'product_residual_caveat': 'the local product residual is not a clean convergence test because reduced G_x is mixed and need not square to identity even when stationary',
        'sampling_caveat': '10 trajectories and only the intervals 5-10, 10-20, and 20-50; descriptive evidence only',
    },
    'born_rule_trace_observable': 'tr(C) from the spatial sum of saved cell charge means',
    'born_rule_trace_convention': records[0]['trace_convention'],
    'born_rule_sampling': '100 Monte Carlo trajectories with Born-rule measurement draws; postselect=False',
    'trajectory_trace_variance': 'population variance across the 100 sampled trajectory traces, ddof=0',
    'born_rule_trace_normalization': 'both plotted quantities divided by total single-particle mode count V=2*Nx*Ny; half filling is V/2=Nx*Ny',
    'born_rule_trace_rolling_average': {'window': ROLLING_WINDOW, 'center': ROLLING_CENTER, 'min_periods': ROLLING_MIN_PERIODS},
    'prior_related_analysis': {
        'notebook': str((BUNDLE_ROOT / 'notebooks' / 'characterization' / 'load_purification_charge_fluctuation_data_cpu_edited.ipynb').relative_to(REPO_ROOT)),
        'date_from_artifact_mtime': '2026-05-06',
        'relationship': 'same campaign; centered-charge mean rolling overlays and rolling percent variance for fixed-cycle Ny scaling; different normalization from the current trace/V figure',
    },
    'prior_frobenius_analysis': {
        'notebook': str((REPO_ROOT / '00_WORKSPACE' / 'LARGE_RESULTS' / 'dw_convergence' / 'n20x20_maxmix_dwtrunc_convergence_cpu' / 'n20x20_dwtrunc_convergence_cpu.ipynb').relative_to(REPO_ROOT)),
        'relationship': 'previous N20x20 max-mixed convergence analysis plotted frob_successive_delta_mean; the current figures add three system sizes, raw ||G||_F, its derivative, and the exact pure-state adjacent-cycle product residual',
    },
    'prior_product_metric_analysis': {
        'notebook': str((BUNDLE_ROOT / 'notebooks' / 'characterization' / 'diagnose_perfect_correction_slope_excess_cpu.ipynb').relative_to(REPO_ROOT)),
        'relationship': 'same pure streaming runs previously used frob_successive_delta as a convergence feature and plotted it alongside entropy-slope excess; the current figure exposes its exact product-residual interpretation, includes the full saved cycle range, and adds the absolute cycle derivative',
    },
    'spatial_localization_analysis': {
        'time_window': 'late half: cycles Ny+1 through 2*Ny',
        'walls': [5, 15],
        'active_slab': [5, 15],
        'active_wall_rows': [5, 6, 14, 15],
        'active_bulk_rows': [7, 8, 9, 10, 11, 12, 13],
        'capture_weight': 'sum of each nonnegative late-time y-averaged x profile',
        'prior_estimator_correction': 'the May 6 wall/bulk notebook applied wall/bulk RMS to occupancy n itself; this analysis uses abs(E[n]-1), Var_traj(n), and the separately saved local quantum variance',
        'scope_caveat': 'dw_truncation=True makes the near-zero exterior partly protocol-imposed; localization inside the active slab is descriptive, not a proof of asymptotic confinement',
    },
    'aggregation': 'trajectory arithmetic mean with one-standard-error bands',
    'derivative': 'trajectory mean of absolute numpy.gradient values on saved cycle labels, edge_order=2; charge-variance curve is unsmoothed, while Frobenius-rate and product-residual plots add centered rolling w=5 guides over unsmoothed means',
    'axis_scales': {'charge_variance_y': 'log', 'charge_absolute_derivative_y': 'log', 'frobenius_norm_rms_normalized_y': 'linear', 'frobenius_absolute_derivative_rms_normalized_y': 'log', 'product_residual_y': 'log', 'product_residual_absolute_derivative_y': 'log', 'spatial_reduced_covariance_rms_normalized_y': 'log with 1e-12 display floor', 'cycle_x': 'linear'},
    'geometries': [[record['Nx'], record['Ny']] for record in records],
    'source_files': [
        str(path.relative_to(REPO_ROOT))
        for record in records
        for path in (record['variance_path'], record['charge_trace_path'], record['local_variance_path'])
    ] + [
        str(path.relative_to(REPO_ROOT))
        for record in product_records
        for path in (record['scalar_path'], record['summary_path'])
    ] + [
        str(path.relative_to(REPO_ROOT)) for path in spatial_snapshot_sources
    ] + [
        str(path.relative_to(REPO_ROOT)) for path in purification_sources
    ],
    'outputs': {
        'charge_variance_png': str(png_path.relative_to(REPO_ROOT)),
        'charge_variance_pdf': str(pdf_path.relative_to(REPO_ROOT)),
        'charge_variance_table': str(csv_path.relative_to(REPO_ROOT)),
        'frobenius_norm_png': str(frob_png_path.relative_to(REPO_ROOT)),
        'frobenius_norm_pdf': str(frob_pdf_path.relative_to(REPO_ROOT)),
        'frobenius_norm_table': str(frob_csv_path.relative_to(REPO_ROOT)),
        'frobenius_convergence_summary_table': str(frob_convergence_csv_path.relative_to(REPO_ROOT)),
        'adjacent_cycle_product_residual_png': str(product_png_path.relative_to(REPO_ROOT)),
        'adjacent_cycle_product_residual_pdf': str(product_pdf_path.relative_to(REPO_ROOT)),
        'adjacent_cycle_product_residual_table': str(product_csv_path.relative_to(REPO_ROOT)),
        'adjacent_cycle_product_residual_summary_table': str(product_summary_csv_path.relative_to(REPO_ROOT)),
        'spatial_reduced_covariance_pilot_png': str(spatial_covariance_png_path.relative_to(REPO_ROOT)),
        'spatial_reduced_covariance_pilot_pdf': str(spatial_covariance_pdf_path.relative_to(REPO_ROOT)),
        'spatial_reduced_covariance_profile_table': str(spatial_covariance_profile_csv_path.relative_to(REPO_ROOT)),
        'spatial_reduced_covariance_region_table': str(spatial_covariance_region_csv_path.relative_to(REPO_ROOT)),
        'spatial_reduced_covariance_wall_bulk_rate_ratio_table': str(spatial_covariance_rate_ratio_csv_path.relative_to(REPO_ROOT)),
        'quantitative_transient_png': str(transient_png_path.relative_to(REPO_ROOT)),
        'quantitative_transient_pdf': str(transient_pdf_path.relative_to(REPO_ROOT)),
        'quantitative_global_transient_table': str(transient_table_path.relative_to(REPO_ROOT)),
        'wall_rate_decay_model_comparison_table': str(wall_decay_fit_table_path.relative_to(REPO_ROOT)),
        'wall_bulk_purification_png': str(wall_bulk_png_path.relative_to(REPO_ROOT)),
        'wall_bulk_purification_pdf': str(wall_bulk_pdf_path.relative_to(REPO_ROOT)),
        'wall_bulk_entropy_region_curves_table': str(wall_bulk_region_table_path.relative_to(REPO_ROOT)),
        'wall_bulk_entropy_fit_summary_table': str(wall_bulk_fit_table_path.relative_to(REPO_ROOT)),
        'wall_bulk_entropy_convergence_thresholds_table': str(wall_bulk_threshold_table_path.relative_to(REPO_ROOT)),
        'wall_power_law_window_sensitivity_table': str(wall_bulk_window_table_path.relative_to(REPO_ROOT)),
        'wall_bulk_purification_scaling_summary_table': str(wall_bulk_scaling_table_path.relative_to(REPO_ROOT)),
        'born_rule_trace_png': str(trace_png_path.relative_to(REPO_ROOT)),
        'born_rule_trace_pdf': str(trace_pdf_path.relative_to(REPO_ROOT)),
        'born_rule_trace_summary_table': str(trace_summary_csv_path.relative_to(REPO_ROOT)),
        'born_rule_trace_by_trajectory_table': str(trajectory_trace_csv_path.relative_to(REPO_ROOT)),
        'spatial_localization_png': str(localization_png_path.relative_to(REPO_ROOT)),
        'spatial_localization_pdf': str(localization_pdf_path.relative_to(REPO_ROOT)),
        'spatial_x_profile_table': str(spatial_profile_csv_path.relative_to(REPO_ROOT)),
        'spatial_localization_summary_table': str(localization_csv_path.relative_to(REPO_ROOT)),
    },
    'diagnostics': diagnostics,
}
manifest_path.write_text(json.dumps(manifest, indent=2) + '\n')

display(diagnostics_df)
display(frob_convergence_df)
display(product_summary_df)
display(rate_region_wide)
display(localization_df[[
    'Ny', 'metric', 'capture_fraction_on_wall_rows_r0',
    'capture_fraction_within_wall_distance_1', 'capture_fraction_within_wall_distance_2',
    'wall_distance_for_90_percent_capture', 'capture_fraction_in_active_slab_x5_to_x15',
    'active_wall_to_bulk_mean_ratio',
]])
print(f'Formula: {records[0]["formula"]}')
print(f'Null counts:\n{plot_data_df.isna().sum().to_string()}')
print(f'Frobenius temporal rows: {len(frob_df)}; nulls: {int(frob_df.isna().sum().sum())}')
print(f'Product-residual temporal rows: {len(product_df)}; nulls: {int(product_df.isna().sum().sum())}')
print(f'Trace-summary null counts:\n{trace_summary_df.isna().sum().to_string()}')
print(f'Raw trajectory-trace rows: {len(trajectory_trace_df)}; nulls: {int(trajectory_trace_df.isna().sum().sum())}')
print(f'Spatial-profile rows: {len(spatial_profile_df)}; nulls: {int(spatial_profile_df.isna().sum().sum())}')
print(f'Localization-summary rows: {len(localization_df)}; nulls: {int(localization_df.isna().sum().sum())}')
print(f'Spatial covariance profile rows: {len(spatial_covariance_profile_df)}; nulls: {int(spatial_covariance_profile_df.isna().sum().sum())}')
print(f'Saved table: {csv_path}')
print(f'Saved Frobenius table: {frob_csv_path}')
print(f'Saved Frobenius convergence summary: {frob_convergence_csv_path}')
print(f'Saved product-residual table: {product_csv_path}')
print(f'Saved product-residual summary: {product_summary_csv_path}')
print(f'Saved trace summary: {trace_summary_csv_path}')
print(f'Saved trajectory traces: {trajectory_trace_csv_path}')
print(f'Saved spatial profiles: {spatial_profile_csv_path}')
print(f'Saved localization summary: {localization_csv_path}')
print(f'Saved spatial covariance profiles: {spatial_covariance_profile_csv_path}')
print(f'Saved spatial covariance region summary: {spatial_covariance_region_csv_path}')
print(f'Saved spatial covariance wall/bulk ratios: {spatial_covariance_rate_ratio_csv_path}')
print(f'Saved manifest: {manifest_path}')